In [ ]:
# code used before modifications in inference.py
# import os

# PROJECT_ROOT = "/workspaces/lpbf_project"
# ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")
# os.makedirs(ASSET_DIR, exist_ok=True)

# WEIGHTS_SRC = os.path.join(PROJECT_ROOT, "notebooks", "best_meltpool_surrogate_400.pt")
# WEIGHTS_DST = os.path.join(ASSET_DIR, "weights.pt")

# SCALER_DST  = os.path.join(ASSET_DIR, "scaler.joblib")
# CONFIG_DST  = os.path.join(ASSET_DIR, "surrogate_config.json")

# DATASET_PATH = os.path.join(PROJECT_ROOT, "data", "clean", "meltpool_master_with_VED.csv")

# print("ASSET_DIR:", ASSET_DIR)
# print("WEIGHTS_SRC exists:", os.path.exists(WEIGHTS_SRC))
# print("DATASET exists:", os.path.exists(DATASET_PATH))


ASSET_DIR: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1
WEIGHTS_SRC exists: True
DATASET exists: True


In [1]:
import os, torch
from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
print(" SurrogateInference loaded.")
print("materials:", sur.material_to_id)


Using device: cpu
 SurrogateInference loaded.
materials: {'316L': 0, 'IN718': 1, 'Ti64': 2}


In [2]:
import torch

GRID_SHAPE = (32, 32, 16)      # (X, Y, Z) - keep same as your earlier tests
TILES = (2, 2, 2)             # 2x2x2
K = TILES[0] * TILES[1] * TILES[2]

X, Y, Z = GRID_SHAPE
tx, ty, tz = TILES


tile_x = torch.arange(X) // (X // tx)
tile_y = torch.arange(Y) // (Y // ty)
tile_z = torch.arange(Z) // (Z // tz)

tile_indices = (tile_x[:, None, None] * (ty * tz)
                + tile_y[None, :, None] * tz
                + tile_z[None, None, :]).long().to(device)

print(f"K tiles: {K} ({tx}x{ty}x{tz})")
print("tile_indices shape:", tile_indices.shape, "| min/max:", int(tile_indices.min()), int(tile_indices.max()))


mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]
print(f"Material: {mat_name} (id={mat_id})")


cfg = sur.cfg
P_min, P_max = cfg["validity_window_by_material"][mat_name]["P_W"]
v_min, v_max = cfg["validity_window_by_material"][mat_name]["v_mm_per_s"]
h_const = float(cfg["validity_window_by_material"][mat_name]["h_mm"][0])
t_const = float(cfg["validity_window_by_material"][mat_name]["t_mm"][0])

print(f"P range: [{P_min}, {P_max}] W | v range: [{v_min}, {v_max}] mm/s")
print(f"Fixed h={h_const} mm | t={t_const} mm")


P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

# bounded per-tile values
P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)  # (K,)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)  # (K,)

# broadcast to full grid
P_map = P_tile[tile_indices]
v_map = v_tile[tile_indices]
h_map = torch.full_like(P_map, h_const)
t_map = torch.full_like(P_map, t_const)

print("P_map stats:", float(P_map.min()), float(P_map.max()), "| expected midpoint:", (P_min + P_max) / 2)
print("v_map stats:", float(v_map.min()), float(v_map.max()), "| expected midpoint:", (v_min + v_max) / 2)
print("h_map unique:", torch.unique(h_map).tolist(), "| t_map unique:", torch.unique(t_map).tolist())


K tiles: 8 (2x2x2)
tile_indices shape: torch.Size([32, 32, 16]) | min/max: 0 7
Material: IN718 (id=1)
P range: [100.0, 600.0] W | v range: [400.0, 1100.0] mm/s
Fixed h=0.11 mm | t=0.04 mm
P_map stats: 350.0 350.0 | expected midpoint: 350.0
v_map stats: 750.0 750.0 | expected midpoint: 750.0
h_map unique: [0.10999999940395355] | t_map unique: [0.03999999910593033]


In [3]:
neighbor_pairs = []

def tile_coords(k, tx=2, ty=2, tz=2):
    x = k // (ty * tz)
    y = (k // tz) % ty
    z = k % tz
    return x, y, z

for a in range(K):
    xa, ya, za = tile_coords(a)
    for b in range(K):
        xb, yb, zb = tile_coords(b)
        if abs(xa - xb) + abs(ya - yb) + abs(za - zb) == 1:
            if a < b:
                neighbor_pairs.append((a, b))

print("Neighbor pairs:", neighbor_pairs)
print("Num neighbor pairs:", len(neighbor_pairs))


Neighbor pairs: [(0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3), (2, 6), (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]
Num neighbor pairs: 12


In [4]:
#cell D
import torch

# Fresh learnable raw parameters (K,)
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

# Break symmetry: make tile 0 different from tile 1 (neighbors)
with torch.no_grad():
    P_raw[0] = 0.6
    P_raw[1] = -0.2
    v_raw[0] = 0.3
    v_raw[1] = -0.1

def tv_loss_raw(P_raw, v_raw, neighbor_pairs):
    difP = []
    difV = []
    for a, b in neighbor_pairs:
        difP.append((P_raw[a] - P_raw[b])**2)
        difV.append((v_raw[a] - v_raw[b])**2)
    difP = torch.stack(difP).mean()
    difV = torch.stack(difV).mean()
    return difP + difV, difP, difV

# Compute TV only (no surrogate, no risk)
L_tv, tvP, tvV = tv_loss_raw(P_raw, v_raw, neighbor_pairs)

# Backprop
L_tv.backward()

print("P_raw init:", P_raw.detach().cpu().numpy())
print("v_raw init:", v_raw.detach().cpu().numpy())
print("\nSmoothness loss:", float(L_tv.item()))
print("tvP:", float(tvP.item()), "| tvV:", float(tvV.item()))

print("\nGradients from TV only:")
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

nzP = (P_raw.grad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()
nzV = (v_raw.grad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()
print("\nNonzero tiles (|grad|>1e-9):")
print("P tiles:", nzP)
print("v tiles:", nzV)


P_raw init: [ 0.6 -0.2  0.   0.   0.   0.   0.   0. ]
v_raw init: [ 0.3 -0.1  0.   0.   0.   0.   0.   0. ]

Smoothness loss: 0.14999999105930328
tvP: 0.11999999731779099 | tvV: 0.029999999329447746

Gradients from TV only:
P_raw.grad: [ 0.33333337 -0.20000002 -0.10000001  0.03333334 -0.10000001  0.03333334
  0.          0.        ]
v_raw.grad: [ 0.16666669 -0.10000001 -0.05        0.01666667 -0.05        0.01666667
  0.          0.        ]

Nonzero tiles (|grad|>1e-9):
P tiles: [0, 1, 2, 3, 4, 5]
v tiles: [0, 1, 2, 3, 4, 5]


In [6]:
# Cell E

import torch
import torch.nn.functional as F

# Make sure tile_indices is on device
tile_indices_d = tile_indices.to(device)

# Learnable raw parameters
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

tile_A = 0
mask_A = (tile_indices_d == tile_A)


# Force risky condition ONLY on tile 0: low P, high v
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -6.0   # sigmoid(-6) ~ near min power
    v_raw[tile_A] =  6.0   # sigmoid( 6) ~ near max speed

# Bounds (use IN718 bounds you printed earlier)
P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0
h_const, t_const = 0.11, 0.04

# Build maps
P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)

P_map = P_tile[tile_indices_d]
v_map = v_tile[tile_indices_d]
h_map = torch.full_like(P_map, h_const)
t_map = torch.full_like(P_map, t_const)

# Surrogate prediction (differentiable)
w_um, d_um = sur(P_map, v_map, h_map, t_map, mat_id=sur.material_to_id["IN718"])

# Choose thresholds so LOF definitely activates in tile 0
# (use tile0 depth mean and push d_req above it)
d_tile0 = d_um[mask_A].mean()
d_req = d_tile0 + 50.0
d_max = d_tile0 + 300.0

lof_map = F.relu(d_req - d_um) / (d_req + 1e-12)
key_map = F.relu(d_um - d_max) / (d_max + 1e-12)
risk_map = lof_map + key_map

# --- LOCALIZED reduction (risk only in tile 0) ---
L_risk_local = risk_map[mask_A].mean()

# --- TV in raw space ---
def tv_loss_raw(P_raw, v_raw, neighbor_pairs):
    difP, difV = [], []
    for a, b in neighbor_pairs:
        difP.append((P_raw[a] - P_raw[b])**2)
        difV.append((v_raw[a] - v_raw[b])**2)
    return torch.stack(difP).mean() + torch.stack(difV).mean()

tv_weight = 0.05
L_tv = tv_loss_raw(P_raw, v_raw, neighbor_pairs)

loss = L_risk_local + tv_weight * L_tv

# Backprop
loss.backward()

print("--- Tile-local risk + TV coupling test ---")
print("tile_A =", tile_A)
print("tile0 depth mean:", float(d_tile0.item()))
print("d_req:", float(d_req.item()), "| d_max:", float(d_max.item()))
print("\nLoss terms:")
print("L_risk_local:", float(L_risk_local.item()))
print("L_tv:", float(L_tv.item()))
print("Total:", float(loss.item()))

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("\nTile grads:")
print("P_raw.grad:", Pgrad.numpy())
print("v_raw.grad:", vgrad.numpy())

nzP = (Pgrad.abs() > 1e-6).nonzero(as_tuple=True)[0].tolist()
nzV = (vgrad.abs() > 1e-6).nonzero(as_tuple=True)[0].tolist()
print("\nNonzero tiles (|grad|>1e-6):")
print("P tiles:", nzP)
print("v tiles:", nzV)

print("\nExpected behavior:")
print("- L_risk_local > 0 (LOF activated in tile 0)")
print("- Tile 0 has gradients from physics risk")
print("- Neighbor tiles (1,2,4) may also get gradients due to TV coupling")
print("- Far tiles ideally ~0")


--- Tile-local risk + TV coupling test ---
tile_A = 0
tile0 depth mean: 453.07568359375
d_req: 503.07568359375 | d_max: 753.07568359375

Loss terms:
L_risk_local: 0.09938860684633255
L_tv: 18.0
Total: 0.9993886351585388

Tile grads:
P_raw.grad: [-0.1500309  0.05       0.05       0.         0.05       0.
  0.         0.       ]
v_raw.grad: [ 0.15000714 -0.05       -0.05        0.         -0.05        0.
  0.          0.        ]

Nonzero tiles (|grad|>1e-6):
P tiles: [0, 1, 2, 4]
v tiles: [0, 1, 2, 4]

Expected behavior:
- L_risk_local > 0 (LOF activated in tile 0)
- Tile 0 has gradients from physics risk
- Neighbor tiles (1,2,4) may also get gradients due to TV coupling
- Far tiles ideally ~0


In [ ]:


assert "sur" in globals(), "Run SurrogateInference load cell first"
assert "tile_indices" in globals(), "tile_indices not found"
assert "neighbor_pairs" in globals(), "neighbor_pairs not found"

loss_fn = ProcessRiskLoss(
    sur=sur,
    P_bounds=(100.0, 600.0),     # IN718 bounds
    v_bounds=(400.0, 1100.0),
    h_const=0.11,
    t_const=0.04,
    d_req_um=503.07568359375,    # forced LOF threshold
    d_max_um=753.07568359375,    # forced keyhole threshold
    tv_weight=0.05,
    mask_power=3.0,
)

print(" loss_fn instantiated and ready")


NameError: name 'ProcessRiskLoss' is not defined

In [7]:
import torch


# Step: rho-masked risk smoke test

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use the existing tile grid
tile_A = 0
tile_indices_d = tile_indices_d.to(device)
mask_A = (tile_indices_d == tile_A)

# 1) Build rho: solid ONLY in tile 0, void elsewhere
rho = torch.zeros_like(tile_indices_d, dtype=torch.float32, device=device)
rho[mask_A] = 1.0  # solid in tile 0 only

print("rho stats:", float(rho.min()), float(rho.mean()), float(rho.max()))
print("solid voxels:", int((rho > 0).sum().item()), "/", rho.numel())

# 2) Fresh learnable raw parameters (K tiles)
K = int(tile_indices_d.max().item()) + 1
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

# Force risky condition ONLY in tile 0 (low P, high v)
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -6.0   # near P_min
    v_raw[tile_A] =  6.0   # near v_max

mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]

# Helper to print which tiles got gradients
def print_grad_tiles(tag):
    Pgrad = P_raw.grad.detach().cpu()
    vgrad = v_raw.grad.detach().cpu()
    p_tiles = (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()
    v_tiles = (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()
    print(f"\n[{tag}] Nonzero grad tiles:")
    print("P tiles:", p_tiles)
    print("v tiles:", v_tiles)
    print("P_raw.grad:", Pgrad.numpy())
    print("v_raw.grad:", vgrad.numpy())


# A) TV OFF -> ONLY tile 0 should get gradients

loss_fn.tv_weight = 0.0

P_raw.grad = None
v_raw.grad = None

loss, info = loss_fn(
    rho=rho,
    tile_indices=tile_indices_d,
    P_raw=P_raw,
    v_raw=v_raw,
    neighbor_pairs=neighbor_pairs,
    mat_id=mat_id
)

loss.backward()

print("\n=== RHO-MASK TEST (TV OFF) ===")
print("Total:", float(loss.item()))
print("L_risk:", float(info["L_risk"].item()), "| L_lof:", float(info["L_lof"].item()), "| L_key:", float(info["L_key"].item()))
print("L_tv  :", float(info["L_tv"].item()))
print_grad_tiles("TV OFF")

print("\nExpected (TV OFF): gradients ONLY on tile 0.")


# B) TV ON -> tile 0 + neighbors (1,2,4) should get gradients

loss_fn.tv_weight = 0.05

P_raw.grad = None
v_raw.grad = None

loss2, info2 = loss_fn(
    rho=rho,
    tile_indices=tile_indices_d,
    P_raw=P_raw,
    v_raw=v_raw,
    neighbor_pairs=neighbor_pairs,
    mat_id=mat_id
)

loss2.backward()

print("\n=== RHO-MASK TEST (TV ON) ===")
print("Total:", float(loss2.item()))
print("L_risk:", float(info2["L_risk"].item()), "| L_lof:", float(info2["L_lof"].item()), "| L_key:", float(info2["L_key"].item()))
print("L_tv  :", float(info2["L_tv"].item()))
print_grad_tiles("TV ON")

print("\nExpected (TV ON): tile 0 + its neighbors (usually 1,2,4) get gradients due to TV coupling.")


rho stats: 0.0 0.125 1.0
solid voxels: 2048 / 16384


NameError: name 'loss_fn' is not defined

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

master = pd.read_csv(DATASET_PATH)

numeric_feature_cols = ["P_W", "v_mm_per_s", "LED_J_per_mm", "VED_J_per_mm3", "h_mm", "t_mm"]
target_cols = ["melt_width_um", "melt_depth_um"]

material_to_id = {"316L": 0, "IN718": 1, "Ti64": 2}
mat_id = master["material"].map(material_to_id).values

X_num = master[numeric_feature_cols].values
y = master[target_cols].values

X_num_train, X_num_test, mat_id_train, mat_id_test, y_train, y_test = train_test_split(
    X_num, mat_id, y, test_size=0.2, random_state=42, shuffle=True
)

scaler = StandardScaler()
scaler.fit(X_num_train)

print(" scaler fitted")
print("mean_ (len):", len(scaler.mean_), scaler.mean_)
print("scale_ (len):", len(scaler.scale_), scaler.scale_)


✅ scaler fitted
mean_ (len): 6 [3.13337454e+02 1.19814586e+03 3.21272696e-01 8.66996525e+01
 1.03819530e-01 3.38195303e-02]
scale_ (len): 6 [1.11780670e+02 4.23333961e+02 2.48106718e-01 5.21376594e+01
 4.85865118e-03 4.85865118e-03]


In [3]:
import torch

state_dict = torch.load(WEIGHTS_SRC, map_location="cpu")
torch.save(state_dict, WEIGHTS_DST)

print("✅ weights copied to:", WEIGHTS_DST)
print("File exists:", os.path.exists(WEIGHTS_DST), "| size:", os.path.getsize(WEIGHTS_DST), "bytes")


✅ weights copied to: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/weights.pt
File exists: True | size: 22404 bytes


/tmp/ipykernel_164648/3029602715.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(WEIGHTS_SRC, map_location="cpu")


In [4]:
import joblib

joblib.dump(scaler, SCALER_DST)

print("✅ scaler saved to:", SCALER_DST)
print("File exists:", os.path.exists(SCALER_DST), "| size:", os.path.getsize(SCALER_DST), "bytes")


✅ scaler saved to: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/scaler.joblib
File exists: True | size: 759 bytes


In [5]:
import json
import numpy as np

# global windows
global_window = {
    "P_W": [float(master["P_W"].min()), float(master["P_W"].max())],
    "v_mm_per_s": [float(master["v_mm_per_s"].min()), float(master["v_mm_per_s"].max())],
    "LED_J_per_mm": [float(master["LED_J_per_mm"].min()), float(master["LED_J_per_mm"].max())],
    "VED_J_per_mm3": [float(master["VED_J_per_mm3"].min()), float(master["VED_J_per_mm3"].max())],
    "h_mm": sorted([float(x) for x in master["h_mm"].dropna().unique().tolist()]),
    "t_mm": sorted([float(x) for x in master["t_mm"].dropna().unique().tolist()]),
}

# per-material windows
material_windows = {}
for mat in sorted(master["material"].dropna().unique().tolist()):
    mdf = master[master["material"] == mat]
    material_windows[mat] = {
        "P_W": [float(mdf["P_W"].min()), float(mdf["P_W"].max())],
        "v_mm_per_s": [float(mdf["v_mm_per_s"].min()), float(mdf["v_mm_per_s"].max())],
        "h_mm": sorted([float(x) for x in mdf["h_mm"].dropna().unique().tolist()]),
        "t_mm": sorted([float(x) for x in mdf["t_mm"].dropna().unique().tolist()]),
    }

config = {
    "name": "meltpool_surrogate_v1",
    "weights_file": "weights.pt",
    "scaler_file": "scaler.joblib",
    "numeric_feature_cols": numeric_feature_cols,
    "target_cols": target_cols,
    "output_order": ["width_um", "depth_um"],  # explicit contract
    "material_to_id": material_to_id,
    "num_materials": 3,
    "emb_dim": 3,
    "hidden_sizes": [64, 64],
    "dropout": 0.10,
    "validity_window_global": global_window,
    "validity_window_by_material": material_windows,
    "scaler_mean": [float(x) for x in scaler.mean_.tolist()],
    "scaler_scale": [float(x) for x in scaler.scale_.tolist()],
    "units": {
        "P_W": "W",
        "v_mm_per_s": "mm/s",
        "h_mm": "mm",
        "t_mm": "mm",
        "LED_J_per_mm": "J/mm",
        "VED_J_per_mm3": "J/mm^3",
        "outputs": "um"
    }
}

with open(CONFIG_DST, "w") as f:
    json.dump(config, f, indent=2)

print(" config saved to:", CONFIG_DST)


 config saved to: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/surrogate_config.json


In [6]:
for fn in ["weights.pt", "scaler.joblib", "surrogate_config.json"]:
    p = os.path.join(ASSET_DIR, fn)
    print(fn, "-> exists:", os.path.exists(p), "| size:", os.path.getsize(p) if os.path.exists(p) else None)


weights.pt -> exists: True | size: 22404
scaler.joblib -> exists: True | size: 759
surrogate_config.json -> exists: True | size: 2069


In [7]:
import json
import joblib
import torch
import numpy as np


with open(CONFIG_DST, "r") as f:
    cfg = json.load(f)

scaler_loaded = joblib.load(SCALER_DST)

print(" Config loaded")
print("Numeric features:", cfg["numeric_feature_cols"])
print("Outputs:", cfg["output_order"])
print("Materials:", cfg["material_to_id"])


from torch import nn

class MeltPoolSurrogate(nn.Module):
    def __init__(self, num_numeric_features, num_materials, emb_dim, hidden_sizes, dropout):
        super().__init__()
        self.material_emb = nn.Embedding(num_materials, emb_dim)

        layers = []
        in_dim = num_numeric_features + emb_dim
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 2))

        self.mlp = nn.Sequential(*layers)

    def forward(self, X_num, mat_id):
        emb = self.material_emb(mat_id)
        x = torch.cat([X_num, emb], dim=1)
        return self.mlp(x)

model = MeltPoolSurrogate(
    num_numeric_features=len(cfg["numeric_feature_cols"]),
    num_materials=cfg["num_materials"],
    emb_dim=cfg["emb_dim"],
    hidden_sizes=cfg["hidden_sizes"],
    dropout=cfg["dropout"]
)

state_dict = torch.load(WEIGHTS_DST, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

print("Model rebuilt and weights loaded")


P = 250.0
v = 800.0
h = cfg["validity_window_global"]["h_mm"][0]
t = cfg["validity_window_global"]["t_mm"][0]

LED = P / v
VED = P / (v * h * t)

X = np.array([[P, v, LED, VED, h, t]])
X_scaled = scaler_loaded.transform(X)

X_t = torch.tensor(X_scaled, dtype=torch.float32)
mat_id_t = torch.tensor([cfg["material_to_id"]["Ti64"]], dtype=torch.long)


with torch.no_grad():
    pred = model(X_t, mat_id_t).numpy()

print("\n Surrogate Prediction Check")
print("Predicted melt-pool width (µm):", float(pred[0, 0]))
print("Predicted melt-pool depth (µm):", float(pred[0, 1]))


 Config loaded
Numeric features: ['P_W', 'v_mm_per_s', 'LED_J_per_mm', 'VED_J_per_mm3', 'h_mm', 't_mm']
Outputs: ['width_um', 'depth_um']
Materials: {'316L': 0, 'IN718': 1, 'Ti64': 2}
Model rebuilt and weights loaded

 Surrogate Prediction Check
Predicted melt-pool width (µm): 203.4603271484375
Predicted melt-pool depth (µm): 82.47492980957031


/tmp/ipykernel_164648/1278983659.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(WEIGHTS_DST, map_location="cpu")


In [8]:
import os, textwrap

PROJECT_ROOT = "/workspaces/lpbf_project"
OUT_PATH = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1", "inference.py")

code = r'''
import os, json
import joblib
import torch
import torch.nn as nn

class MeltPoolSurrogate(nn.Module):
    """
    Must match your trained architecture exactly.
    """
    def __init__(self, num_numeric_features, num_materials=3, emb_dim=3, hidden_sizes=(64,64), dropout=0.10):
        super().__init__()
        self.material_emb = nn.Embedding(num_materials, emb_dim)

        layers = []
        in_dim = num_numeric_features + emb_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 2))
        self.mlp = nn.Sequential(*layers)

    def forward(self, X_num, mat_id):
        mat_vec = self.material_emb(mat_id)
        x = torch.cat([X_num, mat_vec], dim=1)
        return self.mlp(x)


class SurrogateInference(nn.Module):
    """
    Differentiable wrapper for DL4TO:
    - Loads config + weights + sklearn scaler
    - Uses torch-only scaling (NO scaler.transform)
    - Computes LED & VED in torch
    """
    def __init__(self, asset_dir: str, device: str | torch.device = "cpu"):
        super().__init__()
        self.asset_dir = asset_dir
        self.device = torch.device(device)

        # --- Load config ---
        cfg_path = os.path.join(asset_dir, "surrogate_config.json")
        with open(cfg_path, "r") as f:
            self.cfg = json.load(f)

        self.numeric_features = self.cfg["numeric_features"]
        self.outputs = self.cfg["outputs"]
        self.material_to_id = self.cfg["material_to_id"]

        # --- Load scaler and convert mean/std to torch tensors (gradient-safe) ---
        scaler_path = os.path.join(asset_dir, "scaler.joblib")
        scaler = joblib.load(scaler_path)

        mean = torch.tensor(scaler.mean_, dtype=torch.float32, device=self.device)
        scale = torch.tensor(scaler.scale_, dtype=torch.float32, device=self.device)

        # Register as buffers so they move with .to(device) and get saved cleanly
        self.register_buffer("scaler_mean", mean)
        self.register_buffer("scaler_scale", scale)

        # --- Build model + load weights ---
        self.model = MeltPoolSurrogate(
            num_numeric_features=len(self.numeric_features),
            num_materials=len(self.material_to_id),
            emb_dim=self.cfg["model"]["emb_dim"],
            hidden_sizes=tuple(self.cfg["model"]["hidden_sizes"]),
            dropout=self.cfg["model"]["dropout"],
        ).to(self.device)

        w_path = os.path.join(asset_dir, "weights.pt")
        state = torch.load(w_path, map_location=self.device, weights_only=True)
        self.model.load_state_dict(state)

        # Freeze weights (but keep graph for inputs!)
        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, P, v, h, t, mat_id: int | torch.Tensor):
        """
        Supports scalars or tensors for P,v,h,t.
        Returns: width_um, depth_um (same shape as P/v)
        """

        # Convert inputs to tensors on device
        P = torch.as_tensor(P, dtype=torch.float32, device=self.device)
        v = torch.as_tensor(v, dtype=torch.float32, device=self.device)
        h = torch.as_tensor(h, dtype=torch.float32, device=self.device)
        t = torch.as_tensor(t, dtype=torch.float32, device=self.device)

        # Avoid divide-by-zero
        eps = 1e-12
        v_safe = torch.clamp(v, min=eps)

        # Physics features (torch)
        LED = P / v_safe
        VED = P / (v_safe * h * t + eps)

        # Broadcast everything to a common shape
        # (works for scalars or voxel grids)
        target_shape = torch.broadcast_shapes(P.shape, v.shape, h.shape, t.shape, LED.shape, VED.shape)
        P   = P.expand(target_shape)
        v   = v.expand(target_shape)
        LED = LED.expand(target_shape)
        VED = VED.expand(target_shape)
        h   = h.expand(target_shape)
        t   = t.expand(target_shape)

        # mat_id tensor expanded to same shape
        if isinstance(mat_id, int):
            mat_id = torch.full(target_shape, mat_id, dtype=torch.long, device=self.device)
        else:
            mat_id = torch.as_tensor(mat_id, dtype=torch.long, device=self.device).expand(target_shape)

        # Flatten to (N, 6) for MLP
        X_raw = torch.stack([P, v, LED, VED, h, t], dim=-1).reshape(-1, len(self.numeric_features))
        mat_flat = mat_id.reshape(-1)

        # Torch-only scaling (keeps gradients)
        X_scaled = (X_raw - self.scaler_mean) / self.scaler_scale

        # Predict (N,2) then reshape back
        y = self.model(X_scaled, mat_flat)
        y = y.reshape(*target_shape, 2)

        width = y[..., 0]
        depth = y[..., 1]
        return width, depth
'''

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w") as f:
    f.write(textwrap.dedent(code))

print(" Created:", OUT_PATH)


 Created: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/inference.py


In [9]:
import os, json

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")
cfg_path = os.path.join(ASSET_DIR, "surrogate_config.json")

with open(cfg_path, "r") as f:
    cfg = json.load(f)

print("Config keys:", list(cfg.keys()))
# show a few likely candidates if they exist
for k in ["numeric_features", "numeric_feature_cols", "outputs", "output_order", "material_to_id"]:
    if k in cfg:
        print(f"\n{k}:\n", cfg[k])


Config keys: ['name', 'weights_file', 'scaler_file', 'numeric_feature_cols', 'target_cols', 'output_order', 'material_to_id', 'num_materials', 'emb_dim', 'hidden_sizes', 'dropout', 'validity_window_global', 'validity_window_by_material', 'scaler_mean', 'scaler_scale', 'units']

numeric_feature_cols:
 ['P_W', 'v_mm_per_s', 'LED_J_per_mm', 'VED_J_per_mm3', 'h_mm', 't_mm']

output_order:
 ['width_um', 'depth_um']

material_to_id:
 {'316L': 0, 'IN718': 1, 'Ti64': 2}


In [10]:
import json, os

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")
cfg_path = os.path.join(ASSET_DIR, "surrogate_config.json")

with open(cfg_path, "r") as f:
    cfg = json.load(f)

# Fix numeric_features key (map from your existing name)
if "numeric_features" not in cfg:
    if "numeric_feature_cols" in cfg:
        cfg["numeric_features"] = cfg["numeric_feature_cols"]
    elif "features" in cfg:
        cfg["numeric_features"] = cfg["features"]
    else:
        raise KeyError("Could not find numeric feature list in config. Expected numeric_feature_cols or similar.")

# Fix outputs key (map from your existing name)
if "outputs" not in cfg:
    if "output_order" in cfg:
        cfg["outputs"] = cfg["output_order"]
    elif "target_cols" in cfg:
        cfg["outputs"] = cfg["target_cols"]
    else:
        raise KeyError("Could not find outputs list in config. Expected output_order or similar.")

with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

print(" Patched surrogate_config.json")
print("numeric_features =", cfg["numeric_features"])
print("outputs          =", cfg["outputs"])


 Patched surrogate_config.json
numeric_features = ['P_W', 'v_mm_per_s', 'LED_J_per_mm', 'VED_J_per_mm3', 'h_mm', 't_mm']
outputs          = ['width_um', 'depth_um']


In [11]:
import os
import torch

# ---- Paths ----
PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- Import the wrapper we just wrote ----
from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference

# ---- Load surrogate inference module ----
sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
print(" SurrogateInference loaded.")


# Simple forward prediction

# Choose a material (example: IN718)
mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]

P = torch.tensor(200.0, device=device)   # W
v = torch.tensor(800.0, device=device)   # mm/s
h = torch.tensor(0.11, device=device)    # mm
t = torch.tensor(0.04, device=device)    # mm

w_pred, d_pred = sur(P, v, h, t, mat_id)
print(f"\nPred check ({mat_name}): width={w_pred.item():.3f} µm, depth={d_pred.item():.3f} µm")


# Gradient smoke test (CRITICAL for TO)

# Make P and v learnable (this simulates TO updating them)
P_learn = torch.tensor(200.0, device=device, requires_grad=True)
v_learn = torch.tensor(800.0, device=device, requires_grad=True)

w, d = sur(P_learn, v_learn, h, t, mat_id)

# Dummy loss: "make depth larger" (any differentiable scalar works)
loss = -d.mean()
loss.backward()

print("\n Gradient smoke test:")
print("d(loss)/dP =", P_learn.grad.item())
print("d(loss)/dv =", v_learn.grad.item())

if P_learn.grad is None or v_learn.grad is None:
    print(" FAIL: gradients are None (graph broken).")
else:
    # They should be non-zero in almost all normal cases
    ok = (abs(P_learn.grad.item()) > 1e-12) and (abs(v_learn.grad.item()) > 1e-12)
    print(" PASS: gradients flow through surrogate." if ok else " WARNING: gradients are ~0; try different P/v point.")


Using device: cpu


KeyError: 'model'

In [ ]:
import lpbf_to.surrogates.meltpool_v1.inference as inf
print("IMPORTING FROM:", inf.__file__)


IMPORTING FROM: /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/inference.py


In [ ]:
import os
import torch

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference

sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
print(" SurrogateInference loaded.")

mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]

P = torch.tensor(200.0, device=device)
v = torch.tensor(800.0, device=device)
h = torch.tensor(0.11, device=device)
t = torch.tensor(0.04, device=device)

w_pred, d_pred = sur(P, v, h, t, mat_id)
print(f"Pred check ({mat_name}): width={w_pred.item():.3f} µm, depth={d_pred.item():.3f} µm")

# Gradient smoke test
P_learn = torch.tensor(200.0, device=device, requires_grad=True)
v_learn = torch.tensor(800.0, device=device, requires_grad=True)

w, d = sur(P_learn, v_learn, h, t, mat_id)
loss = -d.mean()
loss.backward()

print("d(loss)/dP =", P_learn.grad.item())
print("d(loss)/dv =", v_learn.grad.item())


Using device: cpu
 SurrogateInference loaded.
Pred check (IN718): width=160.560 µm, depth=473.175 µm
d(loss)/dP = -0.1377665251493454
d(loss)/dv = 0.029851479455828667


In [ ]:
#  Sanity Check Cell: assets + config + model + scaling + gradients + range checks
# Run this cell in your current packaging notebook after you have SurrogateInference working.

import os, json
import numpy as np
import torch

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

cfg_path   = os.path.join(ASSET_DIR, "surrogate_config.json")
w_path     = os.path.join(ASSET_DIR, "weights.pt")
scaler_path= os.path.join(ASSET_DIR, "scaler.joblib")

print("=== 0) FILE EXISTENCE CHECK ===")
for p in [cfg_path, w_path, scaler_path]:
    print(("checked" if os.path.exists(p) else "❌"), p)

assert os.path.exists(cfg_path), "Missing surrogate_config.json"
assert os.path.exists(w_path), "Missing weights.pt"
assert os.path.exists(scaler_path), "Missing scaler.joblib"

print("\n=== 1) CONFIG CONTENT CHECK ===")
with open(cfg_path, "r") as f:
    cfg = json.load(f)

required_keys = [
    "numeric_features", "outputs", "material_to_id",
    "emb_dim", "hidden_sizes", "dropout",
    "validity_window_global", "validity_window_by_material",
    "scaler_mean", "scaler_scale"
]
missing = [k for k in required_keys if k not in cfg]
print("Missing keys:", missing if missing else "None ")
assert not missing, f"Config missing keys: {missing}"

print("numeric_features:", cfg["numeric_features"])
print("outputs:", cfg["outputs"])
print("materials:", cfg["material_to_id"])
print("emb_dim:", cfg["emb_dim"], "| hidden_sizes:", cfg["hidden_sizes"], "| dropout:", cfg["dropout"])

# Basic shape sanity
nf = cfg["numeric_features"]
assert len(nf) == 6, f"Expected 6 numeric features, got {len(nf)}"
assert cfg["outputs"] == ["width_um", "depth_um"], "Output order mismatch; expected ['width_um','depth_um']"
assert len(cfg["scaler_mean"]) == len(nf) and len(cfg["scaler_scale"]) == len(nf), "Scaler stats length mismatch"

# Validity window sanity
vw = cfg["validity_window_global"]
assert "P_W" in vw and "v_mm_per_s" in vw, "Validity window missing P_W or v_mm_per_s"
print("Global P range:", vw["P_W"], "W")
print("Global v range:", vw["v_mm_per_s"], "mm/s")

print("\n=== 2) LOAD WRAPPER + FORWARD PREDICTION CHECK ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference
sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
sur.eval()
print(" SurrogateInference instantiated.")

# Run one point per material inside validity window (using per-material h,t from config)
def pick_safe_point(mat_name):
    mw = cfg["validity_window_by_material"][mat_name]
    Pmin, Pmax = mw["P_W"]
    vmin, vmax = mw["v_mm_per_s"]
    h = mw["h_mm"][0]
    t = mw["t_mm"][0]
    # choose a mid-point
    P = 0.5*(Pmin+Pmax)
    v = 0.5*(vmin+vmax)
    return P, v, h, t

for mat_name, mat_id in cfg["material_to_id"].items():
    P, v, h, t = pick_safe_point(mat_name)
    P_t = torch.tensor(P, device=device)
    v_t = torch.tensor(v, device=device)
    h_t = torch.tensor(h, device=device)
    t_t = torch.tensor(t, device=device)

    with torch.no_grad():
        w_pred, d_pred = sur(P_t, v_t, h_t, t_t, mat_id)

    print(f"{mat_name:>5} | P={P:.1f}, v={v:.1f}, h={h:.2f}, t={t:.2f}  ->  "
          f"width={w_pred.item():.3f} µm, depth={d_pred.item():.3f} µm")

print("\n=== 3) GRADIENT (BACKPROP) SMOKE TEST ===")
# pick IN718 by default (exists)
mat_name = "IN718" if "IN718" in cfg["material_to_id"] else list(cfg["material_to_id"].keys())[0]
mat_id = cfg["material_to_id"][mat_name]
P0, v0, h0, t0 = pick_safe_point(mat_name)

P_learn = torch.tensor(P0, device=device, requires_grad=True)
v_learn = torch.tensor(v0, device=device, requires_grad=True)
h_t = torch.tensor(h0, device=device)
t_t = torch.tensor(t0, device=device)

w, d = sur(P_learn, v_learn, h_t, t_t, mat_id)
loss = (w.mean() + d.mean())  # any differentiable scalar
loss.backward()

print("Material:", mat_name)
print("d(loss)/dP:", None if P_learn.grad is None else float(P_learn.grad.item()))
print("d(loss)/dv:", None if v_learn.grad is None else float(v_learn.grad.item()))
assert P_learn.grad is not None and v_learn.grad is not None, "❌ Gradients are None -> graph broken"
print(" Gradients exist (graph is intact).")

print("\n=== 4) QUICK RANGE / STABILITY CHECK ===")
# Test that passing extreme-valid points doesn't crash and returns finite values
mw = cfg["validity_window_by_material"][mat_name]
Pmin, Pmax = mw["P_W"]; vmin, vmax = mw["v_mm_per_s"]; h = mw["h_mm"][0]; t = mw["t_mm"][0]
test_points = [
    (Pmin, vmin), (Pmin, vmax),
    (Pmax, vmin), (Pmax, vmax)
]
for (P, v) in test_points:
    P_t = torch.tensor(P, device=device)
    v_t = torch.tensor(v, device=device)
    h_t = torch.tensor(h, device=device)
    t_t = torch.tensor(t, device=device)
    with torch.no_grad():
        w_pred, d_pred = sur(P_t, v_t, h_t, t_t, mat_id)
    ok = torch.isfinite(w_pred).item() and torch.isfinite(d_pred).item()
    print(f"P={P:7.1f}, v={v:7.1f} -> width={w_pred.item():9.3f}, depth={d_pred.item():9.3f} | finite={ok}")
    assert ok, " Non-finite prediction (nan/inf) at a validity-window corner"

print("\n SANITY CHECK PASSED: assets + config + wrapper + gradients + stability are OK.")


=== 0) FILE EXISTENCE CHECK ===
checked /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/surrogate_config.json
checked /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/weights.pt
checked /workspaces/lpbf_project/lpbf_to/surrogates/meltpool_v1/scaler.joblib

=== 1) CONFIG CONTENT CHECK ===
Missing keys: None 
numeric_features: ['P_W', 'v_mm_per_s', 'LED_J_per_mm', 'VED_J_per_mm3', 'h_mm', 't_mm']
outputs: ['width_um', 'depth_um']
materials: {'316L': 0, 'IN718': 1, 'Ti64': 2}
emb_dim: 3 | hidden_sizes: [64, 64] | dropout: 0.1
Global P range: [100.0, 600.0] W
Global v range: [400.0, 1700.0] mm/s

=== 2) LOAD WRAPPER + FORWARD PREDICTION CHECK ===
Using device: cpu
 SurrogateInference instantiated.
 316L | P=350.0, v=750.0, h=0.11, t=0.04  ->  width=194.214 µm, depth=495.721 µm
IN718 | P=350.0, v=750.0, h=0.11, t=0.04  ->  width=196.198 µm, depth=492.469 µm
 Ti64 | P=285.0, v=1500.0, h=0.10, t=0.03  ->  width=158.074 µm, depth=70.501 µm

=== 3) GRADIENT (BACKPROP) SMOKE TEST 

In [ ]:
import torch

MAT_NAME = "IN718"   # change to "Ti64" later when you test Ti64-only
mat_id = sur.material_to_id[MAT_NAME]

# Pull min/max from config (global window; for single-material you can also use validity_window_by_material)
P_min, P_max = sur.cfg["validity_window_global"]["P_W"]
v_min, v_max = sur.cfg["validity_window_global"]["v_mm_per_s"]

# If we want strictly material-specific bounds, uncomment:
# P_min, P_max = sur.cfg["validity_window_by_material"][MAT_NAME]["P_W"]
# v_min, v_max = sur.cfg["validity_window_by_material"][MAT_NAME]["v_mm_per_s"]

print(f"Material: {MAT_NAME} (id={mat_id})")
print(f"P range: [{P_min}, {P_max}] W | v range: [{v_min}, {v_max}] mm/s")

device = next(sur.parameters()).device

# ---- helpers ----
def sigmoid_bound(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

def logit(x):
    return torch.log(x / (1.0 - x))

def init_raw_from_value(x_start, lo, hi, eps=1e-6):
    # alpha in (0,1)
    alpha = (x_start - lo) / (hi - lo)
    alpha = torch.clamp(alpha, eps, 1 - eps)
    return logit(alpha)


INIT_MODE = "mid"   # "mid" or "custom"

if INIT_MODE == "mid":
    # Option A: midpoint init (sigma(0)=0.5)
    P_raw = torch.zeros((), device=device, requires_grad=True)
    v_raw = torch.zeros((), device=device, requires_grad=True)
    print("Init: midpoint (raw=0)")

else:
    # Option B: custom init using logit
    P_start = 350.0
    v_start = 750.0
    P_raw = init_raw_from_value(torch.tensor(P_start, device=device), P_min, P_max).requires_grad_()
    v_raw = init_raw_from_value(torch.tensor(v_start, device=device), v_min, v_max).requires_grad_()
    print(f"Init: custom start P={P_start}, v={v_start}")

# bounded physical parameters
P = sigmoid_bound(P_raw, P_min, P_max)
v = sigmoid_bound(v_raw, v_min, v_max)

# Use typical constants (for IN718/316L use 0.11/0.04, for Ti64 use 0.10/0.03)
if MAT_NAME in ["IN718", "316L"]:
    h = torch.tensor(0.11, device=device)
    t = torch.tensor(0.04, device=device)
else:
    h = torch.tensor(0.10, device=device)
    t = torch.tensor(0.03, device=device)

# ---- forward pass through surrogate ----
w, d = sur(P, v, h, t, mat_id)

print("\nBounded parameter values (after sigmoid):")
print("P =", float(P.detach().cpu()), "W")
print("v =", float(v.detach().cpu()), "mm/s")
print("\nSurrogate outputs at init:")
print("width =", float(w.detach().cpu()), "µm")
print("depth =", float(d.detach().cpu()), "µm")


loss = -d.mean()   # "push depth up" dummy objective
loss.backward()

print("\nGradient check (should NOT be None, typically non-zero):")
print("d(loss)/dP_raw =", None if P_raw.grad is None else float(P_raw.grad.detach().cpu()))
print("d(loss)/dv_raw =", None if v_raw.grad is None else float(v_raw.grad.detach().cpu()))

if (P_raw.grad is None) or (v_raw.grad is None):
    print("\n FAIL: gradients are None -> graph broken")
else:
    print("\n PASS: gradients flow into raw bounded parameters (ready for DL4TO integration)")


Material: IN718 (id=1)
P range: [100.0, 600.0] W | v range: [400.0, 1700.0] mm/s
Init: midpoint (raw=0)

Bounded parameter values (after sigmoid):
P = 350.0 W
v = 1050.0 mm/s

Surrogate outputs at init:
width = 166.89158630371094 µm
depth = 484.9294738769531 µm

Gradient check (should NOT be None, typically non-zero):
d(loss)/dP_raw = -12.856733322143555
d(loss)/dv_raw = 7.742568492889404

 PASS: gradients flow into raw bounded parameters (ready for DL4TO integration)


In [ ]:
#
# Setup for K-Tiles (IN718)

import os, json
import torch
import numpy as np

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")
CFG_PATH = os.path.join(ASSET_DIR, "surrogate_config.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load config
with open(CFG_PATH, "r") as f:
    cfg = json.load(f)

# Choose starting material (single-material integration first)
MAT_NAME = "IN718"
MAT_ID = cfg["material_to_id"][MAT_NAME]

# Material-specific validity window (we will use these bounds for sigmoid parameterization)
P_min, P_max = cfg["validity_window_by_material"][MAT_NAME]["P_W"]
v_min, v_max = cfg["validity_window_by_material"][MAT_NAME]["v_mm_per_s"]

# Fixed process constants (use the ones from that material window)
h_vals = cfg["validity_window_by_material"][MAT_NAME]["h_mm"]
t_vals = cfg["validity_window_by_material"][MAT_NAME]["t_mm"]
h0 = float(h_vals[0])
t0 = float(t_vals[0])

print(f"Material: {MAT_NAME} (id={MAT_ID})")
print(f"P range: [{P_min}, {P_max}] W | v range: [{v_min}, {v_max}] mm/s")
print(f"Fixed h={h0} mm | t={t0} mm")

# We need a target design grid shape for mapping (DL4TO uses 3D density grids)
# Choose a small debug grid first (fast + easy to verify)
GRID_SHAPE = (32, 32, 16)  # (nx, ny, nz) debug size
print("Debug GRID_SHAPE:", GRID_SHAPE)


Using device: cpu
Material: IN718 (id=1)
P range: [100.0, 600.0] W | v range: [400.0, 1100.0] mm/s
Fixed h=0.11 mm | t=0.04 mm
Debug GRID_SHAPE: (32, 32, 16)


In [ ]:

# Step 2: K-tiles + bounded P/v maps

import torch

# ---- Choose tiling resolution (start small, debuggable) ----
# 2x2x2 tiles = 8 regions
Kx, Ky, Kz = 2, 2, 2
K = Kx * Ky * Kz
print("K tiles:", K, f"({Kx}x{Ky}x{Kz})")

nx, ny, nz = GRID_SHAPE
assert nx % Kx == 0 and ny % Ky == 0 and nz % Kz == 0, "GRID_SHAPE must be divisible by tile counts."

# ---- Build tile_indices grid (nx, ny, nz) with integer tile id 0..K-1 ----
tile_indices = torch.empty((nx, ny, nz), dtype=torch.long)

sx, sy, sz = nx // Kx, ny // Ky, nz // Kz
tid = 0
for ix in range(Kx):
    for iy in range(Ky):
        for iz in range(Kz):
            tile_indices[ix*sx:(ix+1)*sx, iy*sy:(iy+1)*sy, iz*sz:(iz+1)*sz] = tid
            tid += 1

tile_indices = tile_indices.to(device)
print("tile_indices shape:", tile_indices.shape, "| min/max:", tile_indices.min().item(), tile_indices.max().item())

# ---- Learnable raw parameters per tile ----
# Midpoint initialization: raw=0 -> sigmoid(0)=0.5 -> (min+max)/2
P_raw = torch.zeros((K,), dtype=torch.float32, device=device, requires_grad=True)
v_raw = torch.zeros((K,), dtype=torch.float32, device=device, requires_grad=True)

# ---- Differentiable mapping: gather each voxel's tile id -> pick its raw param ----
# (nx,ny,nz) -> raw maps (nx,ny,nz)
P_raw_map = P_raw[tile_indices]
v_raw_map = v_raw[tile_indices]

# ---- Sigmoid bounded parameterization ----
P_map = P_min + (P_max - P_min) * torch.sigmoid(P_raw_map)
v_map = v_min + (v_max - v_min) * torch.sigmoid(v_raw_map)

print("P_map stats:", float(P_map.min()), float(P_map.max()), "| expected midpoint:", (P_min + P_max) / 2)
print("v_map stats:", float(v_map.min()), float(v_map.max()), "| expected midpoint:", (v_min + v_max) / 2)

# ---- Fixed h/t grids (same shape) ----
h_map = torch.full((nx, ny, nz), h0, dtype=torch.float32, device=device)
t_map = torch.full((nx, ny, nz), t0, dtype=torch.float32, device=device)

print("h_map unique:", torch.unique(h_map).tolist(), "| t_map unique:", torch.unique(t_map).tolist())


K tiles: 8 (2x2x2)
tile_indices shape: torch.Size([32, 32, 16]) | min/max: 0 7
P_map stats: 350.0 350.0 | expected midpoint: 350.0
v_map stats: 750.0 750.0 | expected midpoint: 750.0
h_map unique: [0.10999999940395355] | t_map unique: [0.03999999910593033]


In [ ]:

# Surgical gradient smoke test 


import torch

tile_A = 0
mask_A = (tile_indices == tile_A).float()

# ---- 0) Clear grads safely ----
P_raw.grad = None
v_raw.grad = None

# ---- 1) Rebuild bounded maps INSIDE this cell (fresh autograd graph) ----
# (Use the same bounds you used earlier in your k-tile cell)
P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0

sigP = torch.sigmoid(P_raw)   # shape (K,)
sigV = torch.sigmoid(v_raw)   # shape (K,)

P_tile = P_min + (P_max - P_min) * sigP   # shape (K,)
v_tile = v_min + (v_max - v_min) * sigV   # shape (K,)

# Broadcast tiles -> voxel maps
P_map = P_tile[tile_indices]  # shape (nx, ny, nz)
v_map = v_tile[tile_indices]  # shape (nx, ny, nz)

# ---- 2) Define a loss that depends ONLY on Tile A power ----
loss = (P_map * mask_A).sum()

# ---- 3) Backprop (only once) ----
loss.backward()

print("Loss:", float(loss.item()))
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())

if v_raw.grad is None:
    print("v_raw.grad is None (expected because v_raw not used in loss)")
else:
    print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

print("\nExpected:")
print(f"- P_raw.grad[{tile_A}] should be NON-zero")
print("- All other P_raw grads should be 0")
print("- v_raw.grad can be None here (loss ignores v_map)")


Loss: 716800.0
P_raw.grad: [256000.      0.      0.      0.      0.      0.      0.      0.]
v_raw.grad is None (expected because v_raw not used in loss)

Expected:
- P_raw.grad[0] should be NON-zero
- All other P_raw grads should be 0
- v_raw.grad can be None here (loss ignores v_map)


In [ ]:
# Safe gradient print helper
def safe_grad(x, name):
    if x.grad is None:
        print(f"{name}.grad is None (expected if {name} not used in loss)")
    else:
        print(f"{name}.grad:", x.grad.detach().cpu().numpy())

safe_grad(P_raw, "P_raw")
safe_grad(v_raw, "v_raw")


P_raw.grad: [256000.      0.      0.      0.      0.      0.      0.      0.]
v_raw.grad is None (expected if v_raw not used in loss)


In [ ]:

#  Tile-level smoothness penalty (neighbor differences)


# Tile grid dims (must match how you built tile_indices)
Kx, Ky, Kz = 2, 2, 2
K = Kx * Ky * Kz
assert P_raw.numel() == K and v_raw.numel() == K

def tile_id(ix, iy, iz):
    return ix + Kx * (iy + Ky * iz)

# Build neighbor pairs (6-neighborhood)
pairs = []
for iz in range(Kz):
    for iy in range(Ky):
        for ix in range(Kx):
            a = tile_id(ix, iy, iz)
            if ix + 1 < Kx: pairs.append((a, tile_id(ix+1, iy, iz)))
            if iy + 1 < Ky: pairs.append((a, tile_id(ix, iy+1, iz)))
            if iz + 1 < Kz: pairs.append((a, tile_id(ix, iy, iz+1)))

pairs = torch.tensor(pairs, dtype=torch.long, device=device)
a_idx = pairs[:, 0]
b_idx = pairs[:, 1]

# Example smoothness loss (L2 on neighbor differences)
lambda_s = 1e-3  # start tiny; you’ll tune later

smooth_P = ((P_raw[a_idx] - P_raw[b_idx])**2).mean()
smooth_v = ((v_raw[a_idx] - v_raw[b_idx])**2).mean()
L_smooth = lambda_s * (smooth_P + smooth_v)

print("Neighbor pairs:", pairs.shape[0])
print("Smoothness loss (raw space):", float(L_smooth.item()))
print("smooth_P:", float(smooth_P.item()), "| smooth_v:", float(smooth_v.item()))


Neighbor pairs: 12
Smoothness loss (raw space): 0.0
smooth_P: 0.0 | smooth_v: 0.0


In [ ]:

# Surrogate-in-the-loop gradient test (tile-local)


import torch

tile_A = 0
mask_A = (tile_indices == tile_A).float()

# Clear grads
P_raw.grad = None
v_raw.grad = None

# ---- Bounds (use IN718 window for now) ----
P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0

# ---- Build P_map, v_map (fresh graph) ----
P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)

P_map = P_tile[tile_indices]
v_map = v_tile[tile_indices]

# ---- h,t maps (fixed for IN718) ----
h_map = torch.full_like(P_map, 0.11)
t_map = torch.full_like(P_map, 0.04)

# ---- Material map (fixed IN718 everywhere for now) ----
mat_id = 1  # IN718
mat_map = torch.full_like(tile_indices, mat_id, dtype=torch.long)

# ---- Call surrogate (returns width, depth with same grid shape) ----
width_map, depth_map = sur(P_map, v_map, h_map, t_map, mat_map)

# ---- Loss only cares about Tile A depth (example: "increase depth") ----
loss = -(depth_map * mask_A).mean()

loss.backward()

print("Loss:", float(loss.item()))
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

print("\nExpected:")
print(f"- P_raw.grad[{tile_A}] should be NON-zero")
print(f"- v_raw.grad[{tile_A}] should be NON-zero")
print("- Other tiles should be ~0 (since loss only uses Tile A)")


Loss: -61.55858612060547
P_raw.grad: [-1.6172067  0.         0.         0.         0.         0.
  0.         0.       ]
v_raw.grad: [0.63727957 0.         0.         0.         0.         0.
 0.         0.        ]

Expected:
- P_raw.grad[0] should be NON-zero
- v_raw.grad[0] should be NON-zero
- Other tiles should be ~0 (since loss only uses Tile A)


In [ ]:
import torch


#  Smoothness loss


# Assumes you already have:
# - P_raw, v_raw : shape (K,) with requires_grad=True
# - tile_nx, tile_ny, tile_nz (for K=2x2x2, these are 2,2,2)
# - neighbor_pairs : list of (a,b) tile index pairs (if you already computed it)
#
# If you DON'T have neighbor_pairs yet, this cell will build it.

def build_neighbor_pairs(tile_nx, tile_ny, tile_nz):
    def tid(ix, iy, iz):
        return ix + tile_nx * (iy + tile_ny * iz)

    pairs = []
    for iz in range(tile_nz):
        for iy in range(tile_ny):
            for ix in range(tile_nx):
                a = tid(ix, iy, iz)
                if ix + 1 < tile_nx: pairs.append((a, tid(ix+1, iy, iz)))
                if iy + 1 < tile_ny: pairs.append((a, tid(ix, iy+1, iz)))
                if iz + 1 < tile_nz: pairs.append((a, tid(ix, iy, iz+1)))
    return pairs

# ---- set tile grid (for 2x2x2 -> K=8) ----
tile_nx, tile_ny, tile_nz = 2, 2, 2
neighbor_pairs = build_neighbor_pairs(tile_nx, tile_ny, tile_nz)

print("Neighbor pairs:", neighbor_pairs)
print("Num neighbor pairs:", len(neighbor_pairs))

# ---- helper: smoothness penalty in raw space ----
def smoothness_loss(raw_vec, pairs):
    # sum (raw[a] - raw[b])^2 over neighbors
    loss = 0.0
    for a, b in pairs:
        loss = loss + (raw_vec[a] - raw_vec[b])**2
    return loss / max(1, len(pairs))

# ---- compute smoothness losses ----
smooth_P = smoothness_loss(P_raw, neighbor_pairs)
smooth_v = smoothness_loss(v_raw, neighbor_pairs)

print("\nSmoothness loss components (raw space):")
print("smooth_P:", float(smooth_P.item()))
print("smooth_v:", float(smooth_v.item()))

# ---- smoke test: make a loss that should push ALL tiles toward being equal ----
# We add a tiny extra term that breaks symmetry (otherwise all-zeros init gives 0 smoothness + 0 grads)
# so we can SEE gradients propagate.
eps_break = 1e-3
symmetry_break = eps_break * (P_raw[0] + v_raw[0])

loss = smooth_P + smooth_v + symmetry_break

# IMPORTANT: clear grads before backward
if P_raw.grad is not None: P_raw.grad.zero_()
if v_raw.grad is not None: v_raw.grad.zero_()

loss.backward()

print("\nGradient check (smoothness should create gradients beyond tile 0):")
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

print("\nExpected behavior:")
print("- With smoothness active, multiple tiles should typically get non-zero grads (not just tile 0).")
print("- If everything is initialized exactly equal (all zeros), smoothness is ~0 and grads are ~0.")
print("- The tiny symmetry_break term makes grads visible so we can confirm plumbing is correct.")


Neighbor pairs: [(0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3), (2, 6), (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]
Num neighbor pairs: 12

Smoothness loss components (raw space):
smooth_P: 0.0
smooth_v: 0.0

Gradient check (smoothness should create gradients beyond tile 0):
P_raw.grad: [0.001 0.    0.    0.    0.    0.    0.    0.   ]
v_raw.grad: [0.001 0.    0.    0.    0.    0.    0.    0.   ]

Expected behavior:
- With smoothness active, multiple tiles should typically get non-zero grads (not just tile 0).
- If everything is initialized exactly equal (all zeros), smoothness is ~0 and grads are ~0.
- The tiny symmetry_break term makes grads visible so we can confirm plumbing is correct.


In [ ]:

# REAL TV / SMOOTHNESS SMOKE TEST (K=8 tiles)
# Goal:
# 1) Create a mismatch between neighboring tiles (0 and 1)
# 2) Compute smoothness loss over neighbor pairs
# 3) Verify gradients appear on BOTH tiles (0 and 1), not just one


import torch

# ---- Reset grads (important if you ran previous cells) ----
if P_raw.grad is not None: P_raw.grad.zero_()
if v_raw.grad is not None: v_raw.grad.zero_()

# ---- Break symmetry: make tile 0 != tile 1 (neighbors) ----
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[0] = 0.6     # tile 0 higher
    P_raw[1] = -0.2    # tile 1 lower
    v_raw[0] = 0.3
    v_raw[1] = -0.1

print("P_raw init:", P_raw.detach().cpu().numpy())
print("v_raw init:", v_raw.detach().cpu().numpy())

# ---- Smoothness (raw-space) over neighbor pairs ----
smooth_P = torch.tensor(0.0, device=device)
smooth_v = torch.tensor(0.0, device=device)

for a, b in neighbor_pairs:
    smooth_P = smooth_P + (P_raw[a] - P_raw[b])**2
    smooth_v = smooth_v + (v_raw[a] - v_raw[b])**2

loss_smooth = smooth_P + smooth_v
print("\nSmoothness loss:", float(loss_smooth.item()))
print("smooth_P:", float(smooth_P.item()), "| smooth_v:", float(smooth_v.item()))

# ----Backprop ----
loss_smooth.backward()

print("\nGradients from smoothness only:")
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

# ---- Expected checks ----
print("\nExpected behavior:")
print("- Tile 0 and Tile 1 should BOTH have non-zero gradients (they differ).")
print("- Neighbors of 0 and 1 may also get non-zero grads (because the graph couples them).")
print("- Tiles far away may stay 0 if they were equal and only indirectly connected.")

# Optional: explicitly highlight tiles 0 and 1
print("\nTile 0 grads -> dL/dP_raw[0] =", float(P_raw.grad[0].item()),
      "| dL/dv_raw[0] =", float(v_raw.grad[0].item()))
print("Tile 1 grads -> dL/dP_raw[1] =", float(P_raw.grad[1].item()),
      "| dL/dv_raw[1] =", float(v_raw.grad[1].item()))


P_raw init: [ 0.6 -0.2  0.   0.   0.   0.   0.   0. ]
v_raw init: [ 0.3 -0.1  0.   0.   0.   0.   0.   0. ]

Smoothness loss: 1.7999999523162842
smooth_P: 1.4399999380111694 | smooth_v: 0.35999998450279236

Gradients from smoothness only:
P_raw.grad: [ 4.  -2.4 -1.2  0.4 -1.2  0.4  0.   0. ]
v_raw.grad: [ 2.  -1.2 -0.6  0.2 -0.6  0.2  0.   0. ]

Expected behavior:
- Tile 0 and Tile 1 should BOTH have non-zero gradients (they differ).
- Neighbors of 0 and 1 may also get non-zero grads (because the graph couples them).
- Tiles far away may stay 0 if they were equal and only indirectly connected.

Tile 0 grads -> dL/dP_raw[0] = 4.0 | dL/dv_raw[0] = 2.0
Tile 1 grads -> dL/dP_raw[1] = -2.4000000953674316 | dL/dv_raw[1] = -1.2000000476837158


In [ ]:
import torch
import torch.nn.functional as F


#  bounded maps + gather (tile -> voxel)

def bounded_from_raw(raw, xmin, xmax):
    # raw: (K,)
    return xmin + (xmax - xmin) * torch.sigmoid(raw)

def build_maps_from_tiles(P_raw, v_raw, tile_indices, P_min, P_max, v_min, v_max):
    P_tiles = bounded_from_raw(P_raw, P_min, P_max)   # (K,)
    v_tiles = bounded_from_raw(v_raw, v_min, v_max)   # (K,)

    # Gather tile values into full grid
    P_map = P_tiles[tile_indices]  # same shape as tile_indices
    v_map = v_tiles[tile_indices]
    return P_map, v_map, P_tiles, v_tiles

def tv_smoothness(raw_vec, neighbor_pairs):
    # Simple squared-difference TV on tile graph
    loss = torch.zeros((), device=raw_vec.device)
    for a, b in neighbor_pairs:
        loss = loss + (raw_vec[a] - raw_vec[b])**2
    return loss


#  Ensure we have neighbor pairs (2x2x2 tile graph -> 12 edges)

if "neighbor_pairs" not in globals():
    neighbor_pairs = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]


#  Build bounded maps

P_map, v_map, P_tiles, v_tiles = build_maps_from_tiles(
    P_raw, v_raw, tile_indices, P_min, P_max, v_min, v_max
)

print("P_map range:", float(P_map.min()), float(P_map.max()))
print("v_map range:", float(v_map.min()), float(v_map.max()))


#  Material + rho-mask

# Start single-material (IN718) 
mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]

# Fake rho field for smoke test (later this will be DL4TO density field)
# Keep it mostly solid so risks actually backprop.
rho = torch.ones_like(P_map, device=P_map.device) * 0.8
mask = rho**3  # common choice


#  Surrogate forward (voxel grid)
# sur returns width, depth with same shape as P_map
width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id)

print("\nSurrogate outputs (sanity):")
print("width_um stats:", float(width_um.min()), float(width_um.mean()), float(width_um.max()))
print("depth_um stats:", float(depth_um.min()), float(depth_um.mean()), float(depth_um.max()))


#  LOF + Keyhole (dimensionless)

# Pick reasonable thresholds for a smoke test.
# (Later you’ll set these based on literature / your assumptions file.)
d_req = 80.0    # required minimum depth (µm) for fusion (example)
d_max = 200.0   # max depth before keyhole risk (example)

d_req_t = torch.tensor(d_req, device=depth_um.device)
d_max_t = torch.tensor(d_max, device=depth_um.device)

LOF = F.relu(d_req_t - depth_um) / d_req_t
KEY = F.relu(depth_um - d_max_t) / d_max_t

process_risk = (LOF + KEY) * mask
L_risk = process_risk.mean()


#  TV/Smoothness on raw tiles (keeps schedule manufacturable)

lambda_tv = 0.05  # start small
L_tv = tv_smoothness(P_raw, neighbor_pairs) + tv_smoothness(v_raw, neighbor_pairs)


#  Total loss + backward

# IMPORTANT: clear old grads to avoid accumulation
if P_raw.grad is not None: P_raw.grad.zero_()
if v_raw.grad is not None: v_raw.grad.zero_()

loss = L_risk + lambda_tv * L_tv
loss.backward()

print("\n--- Loss breakdown ---")
print("L_risk:", float(L_risk.item()))
print("L_tv  :", float(L_tv.item()))
print("Total :", float(loss.item()))


#  Gradient diagnostics: which tiles got hit?

P_g = P_raw.grad.detach().cpu()
v_g = v_raw.grad.detach().cpu()

print("\n--- Tile gradient summary ---")
print("P_raw.grad:", P_g.numpy())
print("v_raw.grad:", v_g.numpy())

print("\nNon-zero grad tiles (|grad| > 1e-9):")
print("P tiles:", (P_g.abs() > 1e-9).nonzero().flatten().tolist())
print("v tiles:", (v_g.abs() > 1e-9).nonzero().flatten().tolist())

# Optional: quick magnitude stats
print("\nGrad stats:")
print("P grad min/max:", float(P_g.min()), float(P_g.max()))
print("v grad min/max:", float(v_g.min()), float(v_g.max()))


P_map range: 325.0830078125 422.8281555175781
v_map range: 732.5145263671875 802.1097412109375

Surrogate outputs (sanity):
width_um stats: 191.486083984375 197.1947784423828 208.88125610351562
depth_um stats: 490.38616943359375 492.9355163574219 498.28546142578125

--- Loss breakdown ---
L_risk: 0.7499145865440369
L_tv  : 1.7999999523162842
Total : 0.8399145603179932

--- Tile gradient summary ---
P_raw.grad: [ 0.20320418 -0.11587087 -0.05585987  0.02414013 -0.05585987  0.02414013
  0.00414013  0.00414013]
v_raw.grad: [ 0.09877972 -0.06160539 -0.03163144  0.00836856 -0.03163144  0.00836856
 -0.00163144 -0.00163144]

Non-zero grad tiles (|grad| > 1e-9):
P tiles: [0, 1, 2, 3, 4, 5, 6, 7]
v tiles: [0, 1, 2, 3, 4, 5, 6, 7]

Grad stats:
P grad min/max: -0.1158708706498146 0.2032041847705841
v grad min/max: -0.061605390161275864 0.09877972304821014


In [ ]:
import torch
import torch.nn.functional as F


# Material-aware process risk loss 
# For first integration: use ONE material everywhere

MAT_NAME = "IN718"   # change to "316L" or "Ti64" later
mat_id = sur.material_to_id[MAT_NAME]

# Build a voxel material-id map (same shape as design grid)
# Assumes tile_indices exists and gives you the domain shape
mat_id_map = torch.full(tile_indices.shape, mat_id, dtype=torch.long, device=device)

# Material-aware depth thresholds (initial engineering assumptions) ----
# These are starting values (you will tune later).
# d_req = minimum depth needed to avoid lack-of-fusion
# d_max = maximum acceptable depth before keyholing risk
# Pick reasonable "starter" thresholds per material (µm).
DEPTH_THRESH = {
    "316L": {"d_req": 120.0, "d_max": 500.0},
    "IN718": {"d_req": 120.0, "d_max": 520.0},
    "Ti64": {"d_req":  80.0, "d_max": 350.0},
}

d_req_val = DEPTH_THRESH[MAT_NAME]["d_req"]
d_max_val = DEPTH_THRESH[MAT_NAME]["d_max"]

d_req = torch.full(tile_indices.shape, d_req_val, dtype=torch.float32, device=device)
d_max = torch.full(tile_indices.shape, d_max_val, dtype=torch.float32, device=device)

print(f"Material={MAT_NAME} | d_req={d_req_val} µm | d_max={d_max_val} µm")

# (Assumes you already have bounds + mapping code from previous steps)


width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id_map)


# LOF: penalize if depth is below required
L_lof = F.relu(d_req - depth_um) / (d_req + 1e-12)

# Keyhole: penalize if depth exceeds max
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)

# Combine (weights optional; keep 1:1 for now)
L_risk_voxel = L_lof + L_key


# If rho_map exists, only penalize solid regions; else use "all-solid" mask.
if "rho_map" in globals():
    solid_mask = rho_map.clamp(0, 1) ** 3
else:
    solid_mask = torch.ones_like(L_risk_voxel)

L_risk = (L_risk_voxel * solid_mask).mean()


# Assumes you already have neighbor_pairs from earlier
def tv_raw_loss(x_raw, neighbor_pairs):
    tv = 0.0
    for a, b in neighbor_pairs:
        tv = tv + (x_raw[a] - x_raw[b]).abs()
    return tv / max(len(neighbor_pairs), 1)

L_tv = tv_raw_loss(P_raw, neighbor_pairs) + tv_raw_loss(v_raw, neighbor_pairs)

# ----  Total loss ----
lambda_tv = 0.05  # keep same as before
loss = L_risk + lambda_tv * L_tv

# ---- Backprop (important: zero grads first!) ----
P_raw.grad = None
v_raw.grad = None
loss.backward()

print("\n--- Loss breakdown ---")
print("L_lof(mean):", float((L_lof * solid_mask).mean().item()))
print("L_key(mean):", float((L_key * solid_mask).mean().item()))
print("L_risk     :", float(L_risk.item()))
print("L_tv       :", float(L_tv.item()))
print("Total      :", float(loss.item()))

print("\n--- Gradient quick check ---")
print("P_raw.grad min/max:", float(P_raw.grad.min().item()), float(P_raw.grad.max().item()))
print("v_raw.grad min/max:", float(v_raw.grad.min().item()), float(v_raw.grad.max().item()))

#  which tiles are being updated
nzP = (P_raw.grad.abs() > 1e-10).nonzero(as_tuple=True)[0].tolist()
nzV = (v_raw.grad.abs() > 1e-10).nonzero(as_tuple=True)[0].tolist()
print("Non-zero grad tiles P:", nzP)
print("Non-zero grad tiles v:", nzV)


Material=IN718 | d_req=120.0 µm | d_max=520.0 µm


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
import torch
import torch.nn.functional as F


# SETTINGS

MAT_NAME = "IN718"
mat_id = sur.material_to_id[MAT_NAME]

DEPTH_THRESH = {
    "316L": {"d_req": 120.0, "d_max": 500.0},
    "IN718": {"d_req": 120.0, "d_max": 520.0},
    "Ti64":  {"d_req":  80.0, "d_max": 350.0},
}

d_req_val = DEPTH_THRESH[MAT_NAME]["d_req"]
d_max_val = DEPTH_THRESH[MAT_NAME]["d_max"]

print(f"Material={MAT_NAME} | d_req={d_req_val} µm | d_max={d_max_val} µm")


P_raw.grad = None
v_raw.grad = None


# Uses same bounds you were using before (global bounds from config)
P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0   # IN718 material window

P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)  # (K,)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)  # (K,)

P_map = P_tile[tile_indices]  # (X,Y,Z)
v_map = v_tile[tile_indices]  # (X,Y,Z)

# h/t are constants (don’t need grad)
h_map = torch.full_like(P_map, 0.11)
t_map = torch.full_like(P_map, 0.04)

# material id map (constant)
mat_id_map = torch.full(tile_indices.shape, mat_id, dtype=torch.long, device=device)


width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id_map)


# Risk terms

d_req = torch.full_like(depth_um, d_req_val)
d_max = torch.full_like(depth_um, d_max_val)

L_lof = F.relu(d_req - depth_um) / (d_req + 1e-12)
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)
L_risk_voxel = L_lof + L_key


if "rho_map" in globals():
    solid_mask = rho_map.clamp(0, 1) ** 3
else:
    solid_mask = torch.ones_like(L_risk_voxel)

L_risk = (L_risk_voxel * solid_mask).mean()


#  TV penalty in RAW tile space

def tv_raw_loss(x_raw, neighbor_pairs):
    tv = 0.0
    for a, b in neighbor_pairs:
        tv = tv + (x_raw[a] - x_raw[b]).abs()
    return tv / max(len(neighbor_pairs), 1)

lambda_tv = 0.05
L_tv = tv_raw_loss(P_raw, neighbor_pairs) + tv_raw_loss(v_raw, neighbor_pairs)

loss = L_risk + lambda_tv * L_tv


#  Backprop ONCE (no error now)

loss.backward()

print("\n--- Loss breakdown ---")
print("L_lof(mean):", float((L_lof * solid_mask).mean().item()))
print("L_key(mean):", float((L_key * solid_mask).mean().item()))
print("L_risk     :", float(L_risk.item()))
print("L_tv       :", float(L_tv.item()))
print("Total      :", float(loss.item()))

print("\n--- Gradient check ---")
print("P_raw.grad min/max:", float(P_raw.grad.min().item()), float(P_raw.grad.max().item()))
print("v_raw.grad min/max:", float(v_raw.grad.min().item()), float(v_raw.grad.max().item()))


NameError: name 'sur' is not defined

In [ ]:
# restarted kernel

In [ ]:
import torch
import torch.nn.functional as F

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:
import os
from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
print("Surrogate loaded")


KeyError: 'model'

In [ ]:
# Grid
GRID_SHAPE = (32, 32, 16)
K = 8  # 2x2x2

tile_indices = torch.zeros(GRID_SHAPE, dtype=torch.long, device=device)

nx, ny, nz = GRID_SHAPE
sx, sy, sz = nx // 2, ny // 2, nz // 2

tid = 0
for ix in range(2):
    for iy in range(2):
        for iz in range(2):
            tile_indices[
                ix*sx:(ix+1)*sx,
                iy*sy:(iy+1)*sy,
                iz*sz:(iz+1)*sz
            ] = tid
            tid += 1

print("Tile index range:", tile_indices.min().item(), tile_indices.max().item())

# Neighbor list (fixed for 2x2x2)
neighbor_pairs = [
    (0,1),(0,2),(0,4),
    (1,3),(1,5),
    (2,3),(2,6),
    (3,7),
    (4,5),(4,6),
    (5,7),
    (6,7)
]


Tile index range: 0 7


In [ ]:
def tv_raw_loss(x_raw, neighbor_pairs):
    loss = 0.0
    for i, j in neighbor_pairs:
        loss = loss + (x_raw[i] - x_raw[j])**2
    return loss


In [ ]:

P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)


# Force LOF condition

with torch.no_grad():
    P_raw[:] = -3.0   # low power
    v_raw[:] = +3.0   # high speed


#  IN718 bounds

P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0

P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)


# Broadcast to grid

P_map = P_tile[tile_indices]
v_map = v_tile[tile_indices]

h_map = torch.full_like(P_map, 0.11)
t_map = torch.full_like(P_map, 0.04)

mat_id_map = torch.full(
    tile_indices.shape,
    sur.material_to_id["IN718"],
    dtype=torch.long,
    device=device
)


#  Surrogate

width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id_map)


#  Risk loss

d_req = torch.full_like(depth_um, 120.0)
d_max = torch.full_like(depth_um, 520.0)

L_lof = F.relu(d_req - depth_um) / d_req
L_key = F.relu(depth_um - d_max) / d_max

L_risk = (L_lof + L_key).mean()


# TV smoothness

lambda_tv = 0.05
L_tv = tv_raw_loss(P_raw, neighbor_pairs) + tv_raw_loss(v_raw, neighbor_pairs)

loss = L_risk + lambda_tv * L_tv
loss.backward()


#  Diagnostics

print("\n--- Risk activation test ---")
print("Depth stats (min/mean/max):",
      depth_um.min().item(),
      depth_um.mean().item(),
      depth_um.max().item())

print("\nLoss terms:")
print("L_lof:", L_lof.mean().item())
print("L_key:", L_key.mean().item())
print("L_risk:", L_risk.item())
print("L_tv:", L_tv.item())
print("Total:", loss.item())

print("\nGradient ranges:")
print("P_raw.grad:", P_raw.grad.min().item(), P_raw.grad.max().item())
print("v_raw.grad:", v_raw.grad.min().item(), v_raw.grad.max().item())


NameError: name 'sur' is not defined

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = "/workspaces/lpbf_project"
ASSET_DIR = os.path.join(PROJECT_ROOT, "lpbf_to", "surrogates", "meltpool_v1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

from lpbf_to.surrogates.meltpool_v1.inference import SurrogateInference
sur = SurrogateInference(asset_dir=ASSET_DIR, device=device).to(device)
print(" SurrogateInference ready")

# Load config for bounds
with open(os.path.join(ASSET_DIR, "surrogate_config.json"), "r") as f:
    cfg = json.load(f)

# Choose ONE material for now (IN718)
mat_name = "IN718"
mat_id = cfg["material_to_id"][mat_name]
vw = cfg["validity_window_by_material"][mat_name]
P_min, P_max = vw["P_W"][0], vw["P_W"][1]
v_min, v_max = cfg["validity_window_global"]["v_mm_per_s"][0], cfg["validity_window_global"]["v_mm_per_s"][1]
h_fixed = float(vw["h_mm"][0])
t_fixed = float(vw["t_mm"][0])

print(f"Material={mat_name} id={mat_id}")
print(f"P bounds: [{P_min}, {P_max}] | v bounds: [{v_min}, {v_max}]")
print(f"h={h_fixed} mm | t={t_fixed} mm")

# Grid + tiles
GRID_SHAPE = (32, 32, 16)
K = 8  # 2x2x2

def make_tile_indices(shape, tiles=(2,2,2), device="cpu"):
    nx, ny, nz = shape
    tx, ty, tz = tiles
    ix = torch.arange(nx, device=device) * tx // nx
    iy = torch.arange(ny, device=device) * ty // ny
    iz = torch.arange(nz, device=device) * tz // nz
    IX, IY, IZ = torch.meshgrid(ix, iy, iz, indexing="ij")
    tile = IX + tx * IY + (tx * ty) * IZ
    return tile.long()

tile_indices = make_tile_indices(GRID_SHAPE, tiles=(2,2,2), device=device)
print("tile_indices:", tile_indices.shape, "| min/max:", int(tile_indices.min()), int(tile_indices.max()))


Using device: cpu
 SurrogateInference ready
Material=IN718 id=1
P bounds: [100.0, 600.0] | v bounds: [400.0, 1700.0]
h=0.11 mm | t=0.04 mm
tile_indices: torch.Size([32, 32, 16]) | min/max: 0 7


In [ ]:
# Learnable raw params (midpoint init)
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

# Make bounded P_map / v_map from raw via sigmoid (tile-wise)
P_tiles = P_min + (P_max - P_min) * torch.sigmoid(P_raw)   # (K,)
v_tiles = v_min + (v_max - v_min) * torch.sigmoid(v_raw)   # (K,)

P_map = P_tiles[tile_indices]  # (32,32,16)
v_map = v_tiles[tile_indices]  # (32,32,16)

h_map = torch.full(GRID_SHAPE, h_fixed, device=device)
t_map = torch.full(GRID_SHAPE, t_fixed, device=device)

# Surrogate prediction on full grid
width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id)

# ---- FORCE RISK ----
# Set d_req slightly ABOVE predicted mean depth so LOF activates everywhere
d_mean = float(depth_um.mean().item())
d_req = d_mean + 50.0          # force LOF
d_max = d_mean + 200.0         # keep keyhole off for this test

L_lof = F.relu(d_req - depth_um) / (d_req + 1e-12)
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)
L_risk = (L_lof + L_key).mean()

# No TV yet (pure risk activation)
loss = L_risk

# Backprop
P_raw.grad = None
v_raw.grad = None
loss.backward()

print("\n--- Forced risk activation test ---")
print("Depth mean:", d_mean)
print("d_req:", d_req, "| d_max:", d_max)
print("L_lof:", float(L_lof.mean().item()))
print("L_key:", float(L_key.mean().item()))
print("L_risk:", float(L_risk.item()))
print("Total:", float(loss.item()))

print("\nGrad ranges:")
print("P_raw.grad min/max:", float(P_raw.grad.min()), float(P_raw.grad.max()))
print("v_raw.grad min/max:", float(v_raw.grad.min()), float(v_raw.grad.max()))
print("Nonzero tiles (P):", [i for i,g in enumerate(P_raw.grad.detach().cpu().numpy()) if abs(g) > 1e-9])
print("Nonzero tiles (v):", [i for i,g in enumerate(v_raw.grad.detach().cpu().numpy()) if abs(g) > 1e-9])


NameError: name 'h_fixed' is not defined

In [ ]:
import torch
import torch.nn.functional as F



#  Helper: build bounded P_map, v_map from raw tile params

P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1700.0

def bound_param(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

# map raw[K] -> grid via tile_indices
def tile_broadcast(tile_vals, tile_idx):
    # tile_vals: (K,)
    # tile_idx: (X,Y,Z) longs in [0,K-1]
    return tile_vals[tile_idx]


tile_A = 0
mask_A = (tile_indices == tile_A).float()

# fixed assumptions (single material IN718 for now)
mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]
h_map = torch.full(tile_indices.shape, 0.11, device=device)
t_map = torch.full(tile_indices.shape, 0.04, device=device)

P_raw.grad = None
v_raw.grad = None

# Give tile 0 a different starting raw value (creates non-zero TV immediately)
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -0.8   # lower power in tile 0
    v_raw[tile_A] = +0.8   # higher speed in tile 0  -> shallower -> more LOF risk

P_tile = bound_param(P_raw, P_min, P_max)   # (K,)
v_tile = bound_param(v_raw, v_min, v_max)   # (K,)

P_map = tile_broadcast(P_tile, tile_indices)  # (X,Y,Z)
v_map = tile_broadcast(v_tile, tile_indices)  # (X,Y,Z)


# Surrogate prediction on grid

width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id)


# Tile-local risk: only Tile 0 has stricter d_req

d_req_base = 120.0

extra_req = 120.0

d_req_map = torch.full_like(depth_um, d_req_base)
d_req_map = d_req_map + extra_req * mask_A  # only tile 0 gets larger requirement

# Keep keyhole threshold global for now
d_max = 520.0

L_lof = F.relu(d_req_map - depth_um) / (d_req_map + 1e-12)
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)
L_risk_map = L_lof + L_key

# mean risk over ONLY tile 0 region (so gradients should focus there)
L_risk = (L_risk_map * mask_A).sum() / (mask_A.sum() + 1e-12)


neighbor_pairs = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),
                  (4,5),(4,6),(5,7),(6,7)]

tv_P = sum((P_raw[a] - P_raw[b]).pow(2) for a,b in neighbor_pairs) / len(neighbor_pairs)
tv_v = sum((v_raw[a] - v_raw[b]).pow(2) for a,b in neighbor_pairs) / len(neighbor_pairs)

lambda_tv = 0.02  # small weight for this smoke test
L_tv = lambda_tv * (tv_P + tv_v)

# total
loss = L_risk + L_tv
loss.backward()

print("--- Tile-local risk + TV smoke test ---")
print(f"Material={mat_name} | tile_A={tile_A}")
print("Depth stats (tile 0) min/mean/max:",
      float((depth_um[mask_A.bool()]).min().item()),
      float((depth_um[mask_A.bool()]).mean().item()),
      float((depth_um[mask_A.bool()]).max().item()))

print("\nLoss terms:")
print("L_risk:", float(L_risk.item()))
print("L_tv  :", float(L_tv.item()))
print("Total :", float(loss.item()))

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("\nTile grads:")
print("P_raw.grad:", Pgrad.numpy())
print("v_raw.grad:", vgrad.numpy())


def nonzero_tiles(g, tol=1e-6):
    return [i for i,val in enumerate(g.numpy()) if abs(val) > tol]

print("\nNonzero tiles (|grad|>1e-6):")
print("P tiles:", nonzero_tiles(Pgrad))
print("v tiles:", nonzero_tiles(vgrad))

print("\nExpected behavior:")
print("- Tile 0 should have the biggest gradients (risk is only computed on tile 0).")
print("- Neighbor tiles (1,2,4) may get gradients due to TV smoothing.")
print("- Far tiles may stay ~0 or small.")


NameError: name 'sur' is not defined

In [ ]:
import torch
import torch.nn.functional as F


if P_raw.grad is not None:
    P_raw.grad.zero_()
if v_raw.grad is not None:
    v_raw.grad.zero_()

device = P_raw.device
tile_A = 0  # we will activate risk ONLY on tile 0
mask_A = (tile_indices == tile_A).float().to(device)


mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]


# (For IN718 in your config: h=0.11, t=0.04)
h = torch.tensor(0.11, device=device)
t = torch.tensor(0.04, device=device)

# Bounds 
P_min, P_max = cfg["validity_window_global"]["P_W"]
v_min, v_max = cfg["validity_window_global"]["v_mm_per_s"]
P_min = float(P_min); P_max = float(P_max)
v_min = float(v_min); v_max = float(v_max)


# Bounded params per tile
P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)  # (K,)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)  # (K,)

# Broadcast to grid via indexing
P_map = P_tile[tile_indices]  # (nx, ny, nz)
v_map = v_tile[tile_indices]

h_map = h.expand_as(P_map)
t_map = t.expand_as(P_map)


# Make tile 0 "more risky": low power + high speed
# (raw -> sigmoid -> bounded)
P_raw.data[tile_A] = -6.0   # very low -> sigmoid ~0.0025 -> near P_min
v_raw.data[tile_A] = +6.0   # very high -> sigmoid ~0.9975 -> near v_max

# Recompute maps after forcing
P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)
P_map = P_tile[tile_indices]
v_map = v_tile[tile_indices]


with torch.set_grad_enabled(True):
    width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_id)

# Focus on tile 0 only
depth_A = depth_um * mask_A


# Make d_req slightly ABOVE current mean depth in tile 0,
# so ReLU(d_req - depth) becomes >0 for that tile.
depth_mean_A = (depth_A.sum() / (mask_A.sum() + 1e-12)).detach()
d_req = depth_mean_A + 50.0          # force LOF
d_max = depth_mean_A + 300.0         # keep keyhole off for this test

# LOF + Keyhole (normalized)
L_lof = F.relu(d_req - depth_um) / (d_req + 1e-12)
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)

# Compute risk ONLY on tile 0
L_risk = (L_lof * mask_A).mean() + (L_key * mask_A).mean()

# No TV in this step (pure physics-driven gradients)
loss = L_risk


loss.backward()


def nonzero_tiles(grads, thr=1e-9):
    nz = (grads.abs() > thr).nonzero(as_tuple=False).flatten().tolist()
    return nz

print("\n--- Forced risk activation test (tile-local, physics-driven) ---")
print(f"Material={mat_name} | tile_A={tile_A}")
print("Tile 0 forced raw values:")
print("  P_raw[tile0] =", float(P_raw.detach().cpu()[tile_A]))
print("  v_raw[tile0] =", float(v_raw.detach().cpu()[tile_A]))

print("\nTile-0 bounded values (after sigmoid):")
print("  P_tile[tile0] =", float(P_tile.detach().cpu()[tile_A]), "W")
print("  v_tile[tile0] =", float(v_tile.detach().cpu()[tile_A]), "mm/s")

print("\nDepth stats (tile 0):")
depth_min_A = float((depth_um[mask_A.bool()].min()).detach().cpu())
depth_mean_A2 = float((depth_um[mask_A.bool()].mean()).detach().cpu())
depth_max_A = float((depth_um[mask_A.bool()].max()).detach().cpu())
print(f"  min/mean/max = {depth_min_A:.3f} / {depth_mean_A2:.3f} / {depth_max_A:.3f} µm")
print(f"  d_req = {float(d_req.cpu()):.3f} µm | d_max = {float(d_max.cpu()):.3f} µm")

print("\nLoss terms:")
print("  L_lof(tile0 mean):", float((L_lof * mask_A).mean().detach().cpu()))
print("  L_key(tile0 mean):", float((L_key * mask_A).mean().detach().cpu()))
print("  L_risk:", float(L_risk.detach().cpu()))
print("  Total:", float(loss.detach().cpu()))

print("\nGradient check (should hit ONLY tile 0 in this step):")
print("  P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("  v_raw.grad:", v_raw.grad.detach().cpu().numpy())

print("\nNonzero tiles (|grad|>1e-9):")
print("  P tiles:", nonzero_tiles(P_raw.grad))
print("  v tiles:", nonzero_tiles(v_raw.grad))

print("\nExpected behavior:")
print("- L_risk > 0 (LOF activated on tile 0)")
print("- Only tile 0 should have nonzero grads (no TV coupling in this test)")



--- Forced risk activation test (tile-local, physics-driven) ---
Material=IN718 | tile_A=0
Tile 0 forced raw values:
  P_raw[tile0] = -6.0
  v_raw[tile0] = 6.0

Tile-0 bounded values (after sigmoid):
  P_tile[tile0] = 101.23631286621094 W
  v_tile[tile0] = 1696.78564453125 mm/s

Depth stats (tile 0):
  min/mean/max = 442.235 / 442.235 / 442.235 µm
  d_req = 492.235 µm | d_max = 742.235 µm

Loss terms:
  L_lof(tile0 mean): 0.012697183527052402
  L_key(tile0 mean): 0.0
  L_risk: 0.012697183527052402
  Total: 0.012697183527052402

Gradient check (should hit ONLY tile 0 in this step):
  P_raw.grad: [-3.3130225e-05  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  v_raw.grad: [1.1874322e-05 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
 0.0000000e+00 0.0000000e+00 0.0000000e+00]

Nonzero tiles (|grad|>1e-9):
  P tiles: [0]
  v tiles: [0]

Expected behavior:
- L_risk > 0 (LOF activated on tile 0)
- Only tile 0 should have 

In [ ]:
import torch
import torch.nn.functional as F

assert "sur" in globals(), "sur not found. Run the SurrogateInference load cell."
assert "tile_indices" in globals(), "tile_indices not found. Run the K-tiles cell."
assert "neighbor_pairs" in globals(), "neighbor_pairs not found. Run the neighbor_pairs cell."
assert "device" in globals(), "device not found."


K = int(tile_indices.max().item() + 1)
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)


tile_A = 0
with torch.no_grad():
    P_raw[tile_A] = -3.0   # toward P_min
    v_raw[tile_A] =  3.0   # toward v_max


P_min, P_max = 100.0, 600.0
v_min, v_max = 400.0, 1100.0  # use your IN718 window (you printed this earlier)

def bound(x_raw, x_min, x_max):
    return x_min + (x_max - x_min) * torch.sigmoid(x_raw)

P_tile = bound(P_raw, P_min, P_max)  # (K,)
v_tile = bound(v_raw, v_min, v_max)  # (K,)

# Map tiles -> full grids
P_map = P_tile[tile_indices]  # (Z,Y,X) or (32,32,16) depending on your tensor
v_map = v_tile[tile_indices]

# Constants (fixed)
h_map = torch.full_like(P_map, 0.11, dtype=torch.float32, device=device)
t_map = torch.full_like(P_map, 0.04, dtype=torch.float32, device=device)


mat_id = sur.material_to_id["IN718"]
mat_grid = torch.full_like(tile_indices, mat_id, dtype=torch.long, device=device)


width_um, depth_um = sur(P_map, v_map, h_map, t_map, mat_grid)


mask_A = (tile_indices == tile_A).float()

depth_tile0_mean = (depth_um * mask_A).sum() / (mask_A.sum() + 1e-12)

d_req = depth_tile0_mean + 80.0  # make LOF definitely active
d_max = depth_tile0_mean + 500.0 # keep keyhole inactive

L_lof = F.relu(d_req - depth_um) / (d_req + 1e-12)
L_key = F.relu(depth_um - d_max) / (d_max + 1e-12)

# Apply ONLY on tile 0
L_risk = (L_lof * mask_A).sum() / (mask_A.sum() + 1e-12)


def tv_pairs(x, pairs):
    acc = 0.0
    for a, b in pairs:
        acc = acc + (x[a] - x[b]).pow(2)
    return acc / max(1, len(pairs))

L_tv = tv_pairs(P_raw, neighbor_pairs) + tv_pairs(v_raw, neighbor_pairs)

# Weights (keep TV noticeable here)
gamma = 1.0      # risk weight
lambda_tv = 0.05 # smoothness weight

loss = gamma * L_risk + lambda_tv * L_tv


P_raw.grad = None
v_raw.grad = None
loss.backward()

print("\n--- Tile-local risk + TV coupling test ---")
print("Material=IN718 | tile_A=0")
print(f"Tile0 depth mean: {float(depth_tile0_mean.item()):.3f} µm")
print(f"d_req={float(d_req.item()):.3f} | d_max={float(d_max.item()):.3f}")

print("\nLoss terms:")
print("L_risk:", float(L_risk.item()))
print("L_tv  :", float(L_tv.item()))
print("Total :", float(loss.item()))

print("\nTile grads:")
print("P_raw.grad:", P_raw.grad.detach().cpu().numpy())
print("v_raw.grad:", v_raw.grad.detach().cpu().numpy())

nzP = [i for i,g in enumerate(P_raw.grad.detach().cpu().tolist()) if abs(g) > 1e-6]
nzV = [i for i,g in enumerate(v_raw.grad.detach().cpu().tolist()) if abs(g) > 1e-6]
print("\nNonzero tiles (|grad|>1e-6):")
print("P tiles:", nzP)
print("v tiles:", nzV)

print("\nExpected behavior:")
print("- Tile 0 should have the biggest gradients (risk computed only on tile 0).")
print("- Neighbor tiles (1,2,4) should also get gradients due to TV coupling.")
print("- Far tiles may be ~0 or very small.")


AssertionError: sur not found. Run the SurrogateInference load cell.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def tile_broadcast(tile_indices, tile_values):
    return tile_values[tile_indices]

class ProcessRiskLoss(nn.Module):
    def __init__(
        self,
        sur,
        P_bounds, v_bounds,
        h_const, t_const,
        d_req_um, d_max_um,
        tv_weight=0.05,
        mask_power=3.0,
    ):
        super().__init__()
        self.sur = sur

        self.Pmin, self.Pmax = map(float, P_bounds)
        self.vmin, self.vmax = map(float, v_bounds)

        self.h_const = float(h_const)
        self.t_const = float(t_const)

        self.d_req_um = float(d_req_um)
        self.d_max_um = float(d_max_um)

        self.tv_weight = float(tv_weight)
        self.mask_power = float(mask_power)

    def bound(self, raw, lo, hi):
        return lo + (hi - lo) * torch.sigmoid(raw)

    def tv_loss_raw(self, P_raw, v_raw, neighbor_pairs):
        difP, difV = [], []
        for a, b in neighbor_pairs:
            difP.append((P_raw[a] - P_raw[b])**2)
            difV.append((v_raw[a] - v_raw[b])**2)
        difP = torch.stack(difP).mean()
        difV = torch.stack(difV).mean()
        return difP + difV, difP, difV

    def forward(
        self,
        rho,
        tile_indices,
        P_raw,
        v_raw,
        neighbor_pairs,
        mat_id,
        active_mask=None,   
    ):
        # ---- 1) Bound per-tile process parameters ----
        P_tile = self.bound(P_raw, self.Pmin, self.Pmax)  # (K,)
        v_tile = self.bound(v_raw, self.vmin, self.vmax)  # (K,)

        # ---- 2) Broadcast to full grid ----
        P_map = tile_broadcast(tile_indices, P_tile)      # (X,Y,Z)
        v_map = tile_broadcast(tile_indices, v_tile)      # (X,Y,Z)

        h_map = torch.full_like(P_map, self.h_const)
        t_map = torch.full_like(P_map, self.t_const)

        # ---- 3) Surrogate prediction (differentiable) ----
        width_um, depth_um = self.sur(P_map, v_map, h_map, t_map, mat_id)

        # ---- 4) Risk maps (dimensionless) ----
        d_req = torch.as_tensor(self.d_req_um, dtype=depth_um.dtype, device=depth_um.device)
        d_max = torch.as_tensor(self.d_max_um, dtype=depth_um.dtype, device=depth_um.device)

        lof = F.relu(d_req - depth_um) / (d_req + 1e-12)
        key = F.relu(depth_um - d_max) / (d_max + 1e-12)

        # ---- 5) Masking (SOLID ONLY) + OPTIONAL surgical mask ----
        solid_mask = rho.clamp(0, 1) ** self.mask_power  # (X,Y,Z)

        
        if active_mask is not None:
            solid_mask = solid_mask * active_mask.to(dtype=solid_mask.dtype, device=solid_mask.device)

        
        masked_lof = lof * solid_mask
        masked_key = key * solid_mask
        den = solid_mask.sum() + 1e-12

        L_lof = masked_lof.sum() / den
        L_key = masked_key.sum() / den
        L_risk = L_lof + L_key

        
        L_tv, tvP, tvV = self.tv_loss_raw(P_raw, v_raw, neighbor_pairs)

        total = L_risk + self.tv_weight * L_tv

        debug = {
            "L_risk": L_risk.detach(),
            "L_lof": L_lof.detach(),
            "L_key": L_key.detach(),
            "L_tv": L_tv.detach(),
            "tvP": tvP.detach(),
            "tvV": tvV.detach(),
            "depth_mean": depth_um.mean().detach(),
            "P_map_minmax": (P_map.min().detach(), P_map.max().detach()),
            "v_map_minmax": (v_map.min().detach(), v_map.max().detach()),
            "mask_sum": den.detach(),  # helpful for debugging
        }
        return total, debug


In [ ]:
import os
import pandas as pd
import torch


PROJECT_ROOT = "/workspaces/lpbf_project"
DATASET_PATH = os.path.join(PROJECT_ROOT, "data", "clean", "meltpool_master_with_VED.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


df = pd.read_csv(DATASET_PATH)

depth = df["melt_depth_um"].astype(float).dropna().values
width = df["melt_width_um"].astype(float).dropna().values

d_req_um = float(pd.Series(depth).quantile(0.20))   # 20th percentile (shallow cutoff)
d_max_um = float(pd.Series(depth).quantile(0.90))   # 90th percentile (deep cutoff)

print("\nDepth percentiles:")
print("d_req (20th %):", d_req_um)
print("d_max (90th %):", d_max_um)


print("\nDepth stats (dataset): min/mean/max =",
      float(depth.min()), float(depth.mean()), float(depth.max()))


mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]

P_bounds = sur.cfg["validity_window_by_material"][mat_name]["P_W"]
v_bounds = sur.cfg["validity_window_by_material"][mat_name]["v_mm_per_s"]

h_const = sur.cfg["validity_window_by_material"][mat_name]["h_mm"][0]
t_const = sur.cfg["validity_window_by_material"][mat_name]["t_mm"][0]

print(f"\nMaterial={mat_name} (id={mat_id})")
print("P_bounds:", P_bounds, "| v_bounds:", v_bounds)
print("h_const:", h_const, "| t_const:", t_const)


loss_fn = ProcessRiskLoss(
    sur=sur,
    P_bounds=P_bounds,
    v_bounds=v_bounds,
    h_const=h_const,
    t_const=t_const,
    d_req_um=d_req_um,
    d_max_um=d_max_um,
    tv_weight=0.05,     # keep small for now
    mask_power=3.0      # matches typical SIMP p
).to(device)

print("\n ProcessRiskLoss built.")


Using device: cpu

Depth percentiles:
d_req (20th %): 95.84546
d_max (90th %): 555.95714286

Depth stats (dataset): min/mean/max = 14.0 232.04074576522726 1226.0

Material=IN718 (id=1)
P_bounds: [100.0, 600.0] | v_bounds: [400.0, 1100.0]
h_const: 0.11 | t_const: 0.04

 ProcessRiskLoss built.


In [10]:
print(ProcessRiskLoss)


<class '__main__.ProcessRiskLoss'>


In [11]:
loss_fn = ProcessRiskLoss(
    sur=sur,
    P_bounds=(100.0, 600.0),     # IN718
    v_bounds=(400.0, 1100.0),
    h_const=0.11,
    t_const=0.04,
    d_req_um=503.07568359375,    # forced LOF threshold
    d_max_um=753.07568359375,
    tv_weight=0.05,
    mask_power=3.0,
)

print(" ProcessRiskLoss instantiated")


 ProcessRiskLoss instantiated


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- assumes these already exist from earlier cells ----
# sur, loss_fn, tile_indices, neighbor_pairs
# and you want IN718 for now:
mat_id = sur.material_to_id["IN718"]

K = int(tile_indices.max().item() + 1)
P_raw = torch.zeros(K, device=device, requires_grad=True)
v_raw = torch.zeros(K, device=device, requires_grad=True)

tile_A = 0
active_mask = (tile_indices.to(device) == tile_A).to(torch.float32)

# dummy rho: solid only in tile_A (so risk is isolated)
rho = active_mask.clone()

# Force risky condition only on tile 0
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -6.0   # low P
    v_raw[tile_A] =  6.0   # high v

# TV OFF for isolation test
loss_fn.tv_weight = 0.0

loss, info = loss_fn(
    rho=rho,
    tile_indices=tile_indices.to(device),
    P_raw=P_raw,
    v_raw=v_raw,
    neighbor_pairs=neighbor_pairs,
    mat_id=mat_id,
    active_mask=active_mask,
)

loss.backward()

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("=== TEST A: Isolation (TV OFF) ===")
print("Total:", float(loss.item()))
print("L_lof:", float(info["L_lof"].item()), "| L_key:", float(info["L_key"].item()), "| L_risk:", float(info["L_risk"].item()))
print("Nonzero P tiles:", (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())
print("Nonzero v tiles:", (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())


=== TEST A: Isolation (TV OFF) ===
Total: 0.09938860684633255
L_lof: 0.09938860684633255 | L_key: 0.0 | L_risk: 0.09938860684633255
Nonzero P tiles: [0]
Nonzero v tiles: [0]


In [13]:
# ---- TEST B: 
loss_fn.tv_weight = 0.05   # turn TV back on

# clear grads
P_raw.grad = None
v_raw.grad = None

loss, info = loss_fn(
    rho=rho,
    tile_indices=tile_indices.to(device),
    P_raw=P_raw,
    v_raw=v_raw,
    neighbor_pairs=neighbor_pairs,
    mat_id=mat_id,
    active_mask=active_mask,
)

loss.backward()

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("=== TEST B: TV Coupling (TV ON) ===")
print("Total:", float(loss.item()))
print("L_risk:", float(info["L_risk"].item()), "| L_tv:", float(info["L_tv"].item()))
print("Nonzero P tiles:", (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())
print("Nonzero v tiles:", (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())


=== TEST B: TV Coupling (TV ON) ===
Total: 0.9993886351585388
L_risk: 0.09938860684633255 | L_tv: 18.0
Nonzero P tiles: [0, 1, 2, 4]
Nonzero v tiles: [0, 1, 2, 4]


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRID_SHAPE = tile_indices.shape  # (32,32,16)

# Material choice for this phase
mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]  # should be 1

# dummy rho if not present
if "rho" not in globals():
    rho = torch.ones(GRID_SHAPE, device=device)
    print(" rho not found -> using dummy rho=1 everywhere for smoke test")


if P_raw.grad is not None: P_raw.grad.zero_()
if v_raw.grad is not None: v_raw.grad.zero_()


tile_A = 0
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -6.0   # near P_min
    v_raw[tile_A] =  6.0   # near v_max


loss, info = loss_fn(
    P_raw=P_raw,
    v_raw=v_raw,
    tile_indices=tile_indices.to(device),
    neighbor_pairs=neighbor_pairs,
    rho=rho,
    mat_id=mat_id
)

loss.backward()


print("\n--- Risk + gradient smoke test ---")
print("Material:", mat_name, "| mat_id:", mat_id)
print("Total loss:", float(loss.item()))
print("L_lof:", float(info["L_lof"].item()))
print("L_key:", float(info["L_key"].item()))
print("L_risk:", float(info["L_risk"].item()))
print("L_tv:", float(info["L_tv"].item()))

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("\nGrad ranges:")
print("P_raw.grad min/max:", float(Pgrad.min()), float(Pgrad.max()))
print("v_raw.grad min/max:", float(vgrad.min()), float(vgrad.max()))

nonzero_P = (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()
nonzero_v = (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist()

print("\nNonzero grad tiles:")
print("P tiles:", nonzero_P)
print("v tiles:", nonzero_v)

print("\nExpected:")
print("- L_lof should be > 0 (LOF activated on tile 0).")
print("- Tile 0 must have gradients.")
print("- Neighbor tiles may also have gradients if TV couples them.")



--- Risk + gradient smoke test ---
Material: IN718 | mat_id: 1
Total loss: 0.9000000357627869
L_lof: 0.0
L_key: 0.0
L_risk: 0.0
L_tv: 18.0

Grad ranges:
P_raw.grad min/max: -0.15000000596046448 0.05000000447034836
v_raw.grad min/max: -0.05000000447034836 0.15000000596046448

Nonzero grad tiles:
P tiles: [0, 1, 2, 4]
v tiles: [0, 1, 2, 4]

Expected:
- L_lof should be > 0 (LOF activated on tile 0).
- Tile 0 must have gradients.
- Neighbor tiles may also have gradients if TV couples them.


In [ ]:
# Make the loss use your forced thresholds
loss_fn.d_req_um = float(loss_fn.d_req)
loss_fn.d_max_um = float(loss_fn.d_max)

print("Updated thresholds used by loss:")
print("d_req_um:", loss_fn.d_req_um)
print("d_max_um:", loss_fn.d_max_um)


Updated thresholds used by loss:
d_req_um: 503.07568359375
d_max_um: 753.07568359375


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Settings ----
mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]
tile_A = 0


def _get_attr(obj, names):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    raise AttributeError(f"Could not find any of {names} on loss_fn. Add them or rename accordingly.")

P_min = float(_get_attr(loss_fn, ["P_min", "p_min", "Pmin"]))
P_max = float(_get_attr(loss_fn, ["P_max", "p_max", "Pmax"]))
v_min = float(_get_attr(loss_fn, ["v_min", "V_min", "vmin"]))
v_max = float(_get_attr(loss_fn, ["v_max", "V_max", "vmax"]))
h_const = float(_get_attr(loss_fn, ["h_const", "h_mm", "h"]))
t_const = float(_get_attr(loss_fn, ["t_const", "t_mm", "t"]))

tile_indices_d = tile_indices.to(device)

# ---- Helper: raw[K] -> bounded_map[grid] using tile_indices ----
def maps_from_raw(P_raw, v_raw, tile_indices_grid):
    # bounded per-tile values
    P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)   # (K,)
    v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)   # (K,)

    # map to grid
    P_map = P_tile[tile_indices_grid]                         # (X,Y,Z)
    v_map = v_tile[tile_indices_grid]                         # (X,Y,Z)

    # constants to grid
    h_map = torch.full_like(P_map, h_const)
    t_map = torch.full_like(P_map, t_const)
    return P_map, v_map, h_map, t_map

# ---- Clear grads ----
P_raw.grad = None
v_raw.grad = None

# ---- Force risky raw settings on tile 0 ----
with torch.no_grad():
    P_raw[:] = 0.0
    v_raw[:] = 0.0
    P_raw[tile_A] = -6.0   # near P_min
    v_raw[tile_A] =  6.0   # near v_max

# ---- Build maps ----
P_map, v_map, h_map, t_map = maps_from_raw(P_raw, v_raw, tile_indices_d)

# ---- Compute tile-0 depth ----
mask_A = (tile_indices_d == tile_A)
with torch.no_grad():
    w_map, d_map = sur(P_map, v_map, h_map, t_map, mat_id)
d_tile0_mean = d_map[mask_A].mean().item()

# ---- FORCE LOF by shifting thresholds relative to predicted depth ----
margin = 50.0
loss_fn.d_req = float(d_tile0_mean + margin)
loss_fn.d_max = float(d_tile0_mean + 300.0)

print(f"Tile0 depth mean: {d_tile0_mean:.3f} µm")
print(f"FORCED d_req: {loss_fn.d_req:.3f} | d_max: {loss_fn.d_max:.3f}")

# ---- Run loss (now LOF must be > 0) ----
loss, info = loss_fn(
    P_raw=P_raw,
    v_raw=v_raw,
    tile_indices=tile_indices_d,
    neighbor_pairs=neighbor_pairs,
    rho=rho,
    mat_id=mat_id
)

loss.backward()

print("\n--- Forced LOF activation test ---")
print("Total:", float(loss.item()))
print("L_lof:", float(info["L_lof"].item()))
print("L_key:", float(info["L_key"].item()))
print("L_risk:", float(info["L_risk"].item()))
print("L_tv :", float(info["L_tv"].item()))

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()
print("\nNonzero grad tiles:")
print("P tiles:", (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())
print("v tiles:", (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())


Tile0 depth mean: 453.076 µm
FORCED d_req: 503.076 | d_max: 753.076

--- Forced LOF activation test ---
Total: 0.9308725595474243
L_lof: 0.03087254986166954
L_key: 0.0
L_risk: 0.03087254986166954
L_tv : 18.0

Nonzero grad tiles:
P tiles: [0, 1, 2, 3, 4, 5, 6, 7]
v tiles: [0, 1, 2, 3, 4, 5, 6, 7]


In [ ]:
# --- Debug: what d_req / d_max fields exist inside loss_fn? ---
keys = [k for k in dir(loss_fn) if "d_req" in k or "dmax" in k or "d_max" in k]
print("Possible threshold attributes on loss_fn:", keys)

for k in keys:
    try:
        v = getattr(loss_fn, k)
        if torch.is_tensor(v):
            print(f"{k}: tensor {tuple(v.shape)} device={v.device} value(sample)={v.flatten()[0].item()}")
        else:
            print(f"{k}: {v} ({type(v)})")
    except Exception as e:
        print(k, "-> error:", e)


Possible threshold attributes on loss_fn: ['d_max', 'd_max_um', 'd_req', 'd_req_um']
d_max: 753.07568359375 (<class 'float'>)
d_max_um: 555.95714286 (<class 'float'>)
d_req: 503.07568359375 (<class 'float'>)
d_req_um: 95.84546 (<class 'float'>)


In [ ]:
loss_fn.tv_weight = 0.0  # or loss_fn.lambda_tv = 0.0 depending on your code

P_raw.grad = None
v_raw.grad = None
loss, info = loss_fn(P_raw=P_raw, v_raw=v_raw,
                     tile_indices=tile_indices.to(device),
                     neighbor_pairs=neighbor_pairs,
                     rho=rho,
                     mat_id=1)
loss.backward()

print("L_risk:", float(info["L_risk"]))
print("Nonzero P tiles:", (P_raw.grad.abs() > 1e-9).nonzero().flatten().tolist())
print("Nonzero v tiles:", (v_raw.grad.abs() > 1e-9).nonzero().flatten().tolist())


L_risk: 0.03087254986166954
Nonzero P tiles: [0, 1, 2, 3, 4, 5, 6, 7]
Nonzero v tiles: [0, 1, 2, 3, 4, 5, 6, 7]


In [ ]:
# After you build or have tile_indices on device
tile_A = 0
mask_A = (tile_indices.to(device) == tile_A)

print("mask_A voxels:", int(mask_A.sum().item()), " / total:", mask_A.numel())


mask_A voxels: 2048  / total: 16384


In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tile_A = 0
mask_A = (tile_indices.to(device) == tile_A)


P_map = P_min + (P_max - P_min) * torch.sigmoid(P_raw[tile_indices.to(device)])
v_map = v_min + (v_max - v_min) * torch.sigmoid(v_raw[tile_indices.to(device)])

h_map = torch.full_like(P_map, float(h_const), dtype=torch.float32, device=device)
t_map = torch.full_like(P_map, float(t_const), dtype=torch.float32, device=device)

# Surrogate prediction
with torch.enable_grad():
    w_um, d_um = sur(P_map, v_map, h_map, t_map, mat_id=1)

# Risk maps (LOF + Keyhole)
d_req = torch.tensor(float(loss_fn.d_req), device=device)
d_max = torch.tensor(float(loss_fn.d_max), device=device)

L_lof_map = F.relu(d_req - d_um) / (d_req + 1e-12)
L_key_map = F.relu(d_um - d_max) / (d_max + 1e-12)
risk_map = L_lof_map + L_key_map

print("Risk map stats (all): min/mean/max =",
      float(risk_map.min()), float(risk_map.mean()), float(risk_map.max()))

print("Risk map stats (tile0): min/mean/max =",
      float(risk_map[mask_A].min()), float(risk_map[mask_A].mean()), float(risk_map[mask_A].max()))

print("Num risky voxels (risk>0):",
      int((risk_map>0).sum().item()), "/", risk_map.numel())

print("Num risky voxels in tile0:",
      int(((risk_map>0) & mask_A).sum().item()), "/", int(mask_A.sum().item()))


Risk map stats (all): min/mean/max = 0.021084534004330635 0.03087254986166954 0.09938862174749374
Risk map stats (tile0): min/mean/max = 0.09938862174749374 0.09938860684633255 0.09938862174749374
Num risky voxels (risk>0): 16384 / 16384
Num risky voxels in tile0: 2048 / 2048


In [ ]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # x, mask same shape
    num = (x * mask).sum()
    den = mask.sum().clamp_min(eps)
    return num / den


In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


mat_name = "IN718"
mat_id = sur.material_to_id[mat_name]
tile_A = 0

tile_indices_d = tile_indices.to(device)
mask_A = (tile_indices_d == tile_A)


try:
    rho_d = rho.to(device).float()
except NameError:
    rho_d = torch.ones_like(tile_indices_d, dtype=torch.float32, device=device)


mask_power = 3.0
solid_mask = rho_d.clamp(0, 1) ** mask_power


P_tile = P_min + (P_max - P_min) * torch.sigmoid(P_raw)      # (K,)
v_tile = v_min + (v_max - v_min) * torch.sigmoid(v_raw)      # (K,)

P_map = P_tile[tile_indices_d]  # (X,Y,Z)
v_map = v_tile[tile_indices_d]

h_map = torch.full_like(P_map, float(h_const), dtype=torch.float32, device=device)
t_map = torch.full_like(P_map, float(t_const), dtype=torch.float32, device=device)

# -----------------------------
w_um, d_um = sur(P_map, v_map, h_map, t_map, mat_id=mat_id)


d_req = torch.tensor(float(loss_fn.d_req), device=device)
d_max = torch.tensor(float(loss_fn.d_max), device=device)

L_lof_map = F.relu(d_req - d_um) / (d_req + 1e-12)
L_key_map = F.relu(d_um - d_max) / (d_max + 1e-12)
risk_map = L_lof_map + L_key_map


eps = 1e-6
local_mask = (mask_A.float() * solid_mask)  # tile-local + solid-only

L_risk_local = (risk_map * local_mask).sum() / (local_mask.sum().clamp_min(eps))


lambda_tv = 0.05  # set >0 to see coupling (e.g., 0.05)

L_tv = torch.tensor(0.0, device=device)
if lambda_tv > 0:
    for (i, j) in neighbor_pairs:
        L_tv = L_tv + (P_raw[i] - P_raw[j])**2 + (v_raw[i] - v_raw[j])**2

loss = L_risk_local + lambda_tv * L_tv


P_raw.grad = None
v_raw.grad = None
loss.backward()

print("\n=== LOCALIZED LOSS (Tile A only) ===")
print("L_risk_local:", float(L_risk_local.item()))
print("L_tv:", float(L_tv.item()))
print("Total:", float(loss.item()))

Pgrad = P_raw.grad.detach().cpu()
vgrad = v_raw.grad.detach().cpu()

print("\nNonzero grad tiles (|grad|>1e-9):")
print("P tiles:", (Pgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())
print("v tiles:", (vgrad.abs() > 1e-9).nonzero(as_tuple=True)[0].tolist())

print("\nGrad min/max:")
print("P_raw.grad:", float(Pgrad.min()), float(Pgrad.max()))
print("v_raw.grad:", float(vgrad.min()), float(vgrad.max()))



=== LOCALIZED LOSS (Tile A only) ===
L_risk_local: 0.09938860684633255
L_tv: 216.0
Total: 10.899389266967773

Nonzero grad tiles (|grad|>1e-9):
P tiles: [0, 1, 2, 4]
v tiles: [0, 1, 2, 4]

Grad min/max:
P_raw.grad: -1.8003108501434326 0.6000000238418579
v_raw.grad: -0.6000000238418579 1.8000718355178833
